In [1]:
# ============================================================
# ROADFLOOD-VLM
# Notebook 05: Transportation Knowledge Extraction
# ============================================================

from pathlib import Path
import sys
import json
import math
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import osmnx as ox

warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
)

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

current_dir = Path.cwd()

if current_dir.name == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
MAP_DIR = OUTPUT_DIR / "maps"

TKG_DIR = (
    PROCESSED_DIR
    / "transportation_knowledge_graphs"
)

GRAPHML_DIR = TKG_DIR / "graphml"
GPKG_DIR = TKG_DIR / "geopackages"

TRANSPORT_KNOWLEDGE_DIR = (
    PROCESSED_DIR
    / "transportation_knowledge"
)

TOKEN_DIR = TRANSPORT_KNOWLEDGE_DIR / "tokens"
SCENE_PROFILE_DIR = (
    TRANSPORT_KNOWLEDGE_DIR
    / "scene_profiles"
)
CENTRALITY_DIR = (
    TRANSPORT_KNOWLEDGE_DIR
    / "centrality"
)

for directory in [
    TABLE_DIR,
    FIGURE_DIR,
    MAP_DIR,
    TRANSPORT_KNOWLEDGE_DIR,
    TOKEN_DIR,
    SCENE_PROFILE_DIR,
    CENTRALITY_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

# ------------------------------------------------------------
# Input files
# ------------------------------------------------------------

metrics_path = (
    TABLE_DIR
    / "roadflood_vlm_top30_transportation_metrics.csv"
)

processing_log_path = (
    TABLE_DIR
    / "roadflood_vlm_top30_transportation_processing_log.csv"
)

if not metrics_path.exists():
    raise FileNotFoundError(
        "Transportation metrics table was not found. "
        "Run Notebook 04 before continuing."
    )

if not processing_log_path.exists():
    raise FileNotFoundError(
        "Transportation processing log was not found. "
        "Run Notebook 04 before continuing."
    )

print("=" * 72)
print("ROADFLOOD-VLM TRANSPORTATION KNOWLEDGE EXTRACTION")
print("=" * 72)

print(f"\nProject root       : {PROJECT_ROOT}")
print(f"Metrics table      : {metrics_path}")
print(f"GraphML directory  : {GRAPHML_DIR}")
print(f"GeoPackage directory: {GPKG_DIR}")
print(f"Output directory   : {TRANSPORT_KNOWLEDGE_DIR}")

print(f"\nOSMnx version      : {ox.__version__}")
print(f"NetworkX version   : {nx.__version__}")
print(f"GeoPandas version  : {gpd.__version__}")
print(f"Python version     : {sys.version.split()[0]}")

print("\n" + "=" * 72)
print("NOTEBOOK 05 INITIALIZATION COMPLETE")
print("=" * 72)

ROADFLOOD-VLM TRANSPORTATION KNOWLEDGE EXTRACTION

Project root       : /home/adjeiowusu1/myproject/ResilientVLM
Metrics table      : /home/adjeiowusu1/myproject/ResilientVLM/outputs/tables/roadflood_vlm_top30_transportation_metrics.csv
GraphML directory  : /home/adjeiowusu1/myproject/ResilientVLM/data/processed/transportation_knowledge_graphs/graphml
GeoPackage directory: /home/adjeiowusu1/myproject/ResilientVLM/data/processed/transportation_knowledge_graphs/geopackages
Output directory   : /home/adjeiowusu1/myproject/ResilientVLM/data/processed/transportation_knowledge

OSMnx version      : 2.0.7
NetworkX version   : 3.4.2
GeoPandas version  : 1.1.4
Python version     : 3.10.12

NOTEBOOK 05 INITIALIZATION COMPLETE


In [2]:
# ============================================================
# CELL 2: LOAD TRANSPORTATION-USABLE SCENES
# ============================================================

print("=" * 72)
print("LOAD TRANSPORTATION-USABLE SCENES")
print("=" * 72)

transport_metrics_df = pd.read_csv(
    metrics_path
)

processing_log_df = pd.read_csv(
    processing_log_path
)

required_metric_columns = [
    "scene_id",
    "country_prefix",
    "split",
    "quality_rank",
    "node_count",
    "directed_edge_count",
    "total_road_length_km",
    "road_density_km_per_km2",
    "intersection_count",
    "intersection_density_per_km2",
    "road_class_count",
    "bridge_edge_count",
    "transportation_usable",
]

missing_columns = [
    column
    for column in required_metric_columns
    if column not in transport_metrics_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required metric columns: {missing_columns}"
    )

# Convert Boolean values safely if loaded from CSV as text.
if transport_metrics_df["transportation_usable"].dtype == object:
    transport_metrics_df["transportation_usable"] = (
        transport_metrics_df["transportation_usable"]
        .astype(str)
        .str.lower()
        .map(
            {
                "true": True,
                "false": False,
            }
        )
    )

usable_scenes_df = (
    transport_metrics_df.loc[
        transport_metrics_df[
            "transportation_usable"
        ]
        == True
    ]
    .copy()
    .sort_values(
        [
            "road_density_km_per_km2",
            "intersection_density_per_km2",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

print(
    f"\nTotal metric records         : "
    f"{len(transport_metrics_df):,}"
)

print(
    f"Transportation-usable scenes : "
    f"{len(usable_scenes_df):,}"
)

print("\nUSABLE SCENES BY COUNTRY OR REGION")
print("-" * 72)

print(
    usable_scenes_df[
        "country_prefix"
    ]
    .value_counts()
    .to_string()
)

print("\nUSABLE SCENES BY DATASET SPLIT")
print("-" * 72)

print(
    usable_scenes_df[
        "split"
    ]
    .value_counts()
    .to_string()
)

display_columns = [
    "scene_id",
    "country_prefix",
    "split",
    "quality_rank",
    "node_count",
    "directed_edge_count",
    "road_density_km_per_km2",
    "intersection_density_per_km2",
    "bridge_edge_count",
]

print("\nTOP TRANSPORTATION-USABLE SCENES")
print("-" * 72)

print(
    usable_scenes_df[
        display_columns
    ]
    .head(15)
    .round(3)
    .to_string(index=False)
)

print("\n" + "=" * 72)
print("TRANSPORTATION-USABLE SCENES LOADED")
print("=" * 72)

LOAD TRANSPORTATION-USABLE SCENES

Total metric records         : 30
Transportation-usable scenes : 26

USABLE SCENES BY COUNTRY OR REGION
------------------------------------------------------------------------
country_prefix
India        10
USA           5
Spain         3
Sri-Lanka     2
Somalia       2
Nigeria       2
Mekong        1
Ghana         1

USABLE SCENES BY DATASET SPLIT
------------------------------------------------------------------------
split
train         19
validation     4
test           3

TOP TRANSPORTATION-USABLE SCENES
------------------------------------------------------------------------
       scene_id country_prefix      split  quality_rank  node_count  directed_edge_count  road_density_km_per_km2  intersection_density_per_km2  bridge_edge_count
  Spain_7370579          Spain       test            18         961                 2015                   13.581                        30.280                 34
  Spain_1167260          Spain      train         

In [3]:
# ============================================================
# CELL 3: DEFINE TRANSPORTATION KNOWLEDGE CATEGORIES
# ============================================================

print("=" * 72)
print("TRANSPORTATION KNOWLEDGE CATEGORY DEFINITIONS")
print("=" * 72)

ROAD_HIERARCHY_SCORE = {
    "motorway": 6,
    "trunk": 5,
    "primary": 4,
    "secondary": 3,
    "tertiary": 2,
    "unclassified": 1,
    "residential": 1,
    "service": 0,
    "living_street": 0,
    "road": 0,
    "unknown": 0,
}

ROAD_HIERARCHY_GROUP = {
    "motorway": "limited_access",
    "motorway_link": "limited_access",
    "trunk": "major_arterial",
    "trunk_link": "major_arterial",
    "primary": "arterial",
    "primary_link": "arterial",
    "secondary": "collector",
    "secondary_link": "collector",
    "tertiary": "local_collector",
    "tertiary_link": "local_collector",
    "unclassified": "local",
    "residential": "local",
    "service": "service",
    "living_street": "local",
    "road": "other",
    "unknown": "unknown",
}


def normalize_highway_class(value):
    """
    Convert list-like or compound OSM highway values to one
    representative roadway class.
    """

    if value is None:
        return "unknown"

    if isinstance(value, float) and np.isnan(value):
        return "unknown"

    if isinstance(value, (list, tuple, set)):
        candidates = [
            str(item).strip().lower()
            for item in value
        ]

    else:
        text = str(value).strip()

        if text.startswith("[") and text.endswith("]"):
            text = (
                text
                .replace("[", "")
                .replace("]", "")
                .replace("'", "")
                .replace('"', "")
            )

            candidates = [
                item.strip().lower()
                for item in text.split(",")
            ]

        elif "|" in text:
            candidates = [
                item.strip().lower()
                for item in text.split("|")
            ]

        else:
            candidates = [text.lower()]

    hierarchy_order = sorted(
        ROAD_HIERARCHY_SCORE,
        key=ROAD_HIERARCHY_SCORE.get,
        reverse=True,
    )

    for road_class in hierarchy_order:
        if road_class in candidates:
            return road_class

    return (
        candidates[0]
        if candidates
        else "unknown"
    )


def road_hierarchy_score(highway_class):
    """
    Return an ordinal roadway hierarchy score.
    """

    return ROAD_HIERARCHY_SCORE.get(
        highway_class,
        0,
    )


def road_hierarchy_group(highway_class):
    """
    Convert roadway class into a semantic roadway group.
    """

    return ROAD_HIERARCHY_GROUP.get(
        highway_class,
        "other",
    )


def safe_boolean(value):
    """
    Convert mixed Boolean, numeric, or text fields into Boolean.
    """

    if value is None:
        return False

    if isinstance(value, bool):
        return value

    if isinstance(value, float) and np.isnan(value):
        return False

    text = str(value).strip().lower()

    return text not in {
        "",
        "false",
        "no",
        "none",
        "nan",
        "0",
    }


print("\nRoad hierarchy scores:")
print(
    json.dumps(
        ROAD_HIERARCHY_SCORE,
        indent=2,
    )
)

print("\nRoad hierarchy groups:")
print(
    json.dumps(
        ROAD_HIERARCHY_GROUP,
        indent=2,
    )
)

print("\n" + "=" * 72)
print("TRANSPORTATION KNOWLEDGE CATEGORIES READY")
print("=" * 72)

TRANSPORTATION KNOWLEDGE CATEGORY DEFINITIONS

Road hierarchy scores:
{
  "motorway": 6,
  "trunk": 5,
  "primary": 4,
  "secondary": 3,
  "tertiary": 2,
  "unclassified": 1,
  "residential": 1,
  "service": 0,
  "living_street": 0,
  "road": 0,
  "unknown": 0
}

Road hierarchy groups:
{
  "motorway": "limited_access",
  "motorway_link": "limited_access",
  "trunk": "major_arterial",
  "trunk_link": "major_arterial",
  "primary": "arterial",
  "primary_link": "arterial",
  "secondary": "collector",
  "secondary_link": "collector",
  "tertiary": "local_collector",
  "tertiary_link": "local_collector",
  "unclassified": "local",
  "residential": "local",
  "service": "service",
  "living_street": "local",
  "road": "other",
  "unknown": "unknown"
}

TRANSPORTATION KNOWLEDGE CATEGORIES READY


In [4]:
# ============================================================
# CELL 4: LOAD ONE PROTOTYPE TRANSPORTATION GRAPH
# ============================================================

print("=" * 72)
print("PROTOTYPE TRANSPORTATION GRAPH LOADING")
print("=" * 72)

PROTOTYPE_SCENE_ID = "Spain_8565131"

if PROTOTYPE_SCENE_ID not in set(
    usable_scenes_df["scene_id"]
):
    PROTOTYPE_SCENE_ID = (
        usable_scenes_df.iloc[0]["scene_id"]
    )

prototype_graph_path = (
    GRAPHML_DIR
    / f"{PROTOTYPE_SCENE_ID}_transportation_graph.graphml"
)

prototype_gpkg_path = (
    GPKG_DIR
    / f"{PROTOTYPE_SCENE_ID}_transportation_knowledge.gpkg"
)

if not prototype_graph_path.exists():
    raise FileNotFoundError(
        f"GraphML file not found for "
        f"{PROTOTYPE_SCENE_ID}: "
        f"{prototype_graph_path}"
    )

if not prototype_gpkg_path.exists():
    raise FileNotFoundError(
        f"GeoPackage file not found for "
        f"{PROTOTYPE_SCENE_ID}: "
        f"{prototype_gpkg_path}"
    )

prototype_graph = ox.io.load_graphml(
    prototype_graph_path
)

prototype_nodes_gdf = gpd.read_file(
    prototype_gpkg_path,
    layer="nodes",
)

prototype_roads_gdf = gpd.read_file(
    prototype_gpkg_path,
    layer="roads",
)

prototype_metadata_gdf = gpd.read_file(
    prototype_gpkg_path,
    layer="metadata",
)

print(f"\nPrototype scene: {PROTOTYPE_SCENE_ID}")

print(
    f"Graph nodes    : "
    f"{prototype_graph.number_of_nodes():,}"
)

print(
    f"Graph edges    : "
    f"{prototype_graph.number_of_edges():,}"
)

print(
    f"Node features  : "
    f"{len(prototype_nodes_gdf.columns):,}"
)

print(
    f"Road features  : "
    f"{len(prototype_roads_gdf.columns):,}"
)

print("\nROAD ATTRIBUTE COLUMNS")
print("-" * 72)

print(
    sorted(
        prototype_roads_gdf.columns.tolist()
    )
)

print("\nNODE ATTRIBUTE COLUMNS")
print("-" * 72)

print(
    sorted(
        prototype_nodes_gdf.columns.tolist()
    )
)

print("\nSAMPLE ROAD RECORDS")
print("-" * 72)

sample_columns = [
    column
    for column in [
        "highway",
        "highway_class",
        "road_importance",
        "length_m",
        "lanes_numeric",
        "maxspeed_numeric",
        "is_bridge",
        "is_tunnel",
        "is_oneway",
    ]
    if column in prototype_roads_gdf.columns
]

print(
    prototype_roads_gdf[
        sample_columns
    ]
    .head(10)
    .to_string(index=False)
)

print("\n" + "=" * 72)
print("PROTOTYPE GRAPH LOADED SUCCESSFULLY")
print("=" * 72)

PROTOTYPE TRANSPORTATION GRAPH LOADING



Prototype scene: Spain_8565131
Graph nodes    : 449
Graph edges    : 902
Node features  : 11
Road features  : 28

ROAD ATTRIBUTE COLUMNS
------------------------------------------------------------------------
['access', 'bridge', 'geometry', 'highway', 'highway_class', 'highway_clean', 'is_bridge', 'is_oneway', 'is_tunnel', 'junction', 'key', 'lanes', 'lanes_numeric', 'length', 'length_m', 'maxspeed', 'maxspeed_numeric', 'name', 'name_clean', 'oneway', 'osmid', 'ref', 'reversed', 'road_importance', 'scene_id', 'tunnel', 'u', 'v']

NODE ATTRIBUTE COLUMNS
------------------------------------------------------------------------
['geometry', 'highway', 'is_dead_end', 'is_intersection', 'node_degree', 'osmid', 'ref', 'scene_id', 'street_count', 'x', 'y']

SAMPLE ROAD RECORDS
------------------------------------------------------------------------
     highway highway_class  road_importance    length_m  lanes_numeric  maxspeed_numeric  is_bridge  is_tunnel  is_oneway
    motorway      moto

In [5]:
# ============================================================
# CELL 5: PREPARE ROAD HIERARCHY AND INFRASTRUCTURE ATTRIBUTES
# ============================================================

print("=" * 72)
print("ROAD HIERARCHY AND INFRASTRUCTURE PREPARATION")
print("=" * 72)

prototype_roads_gdf = (
    prototype_roads_gdf.copy()
)

# ------------------------------------------------------------
# Normalize roadway class
# ------------------------------------------------------------

source_highway_column = (
    "highway_class"
    if "highway_class"
    in prototype_roads_gdf.columns
    else "highway"
)

prototype_roads_gdf[
    "knowledge_highway_class"
] = (
    prototype_roads_gdf[
        source_highway_column
    ]
    .apply(normalize_highway_class)
)

prototype_roads_gdf[
    "hierarchy_score"
] = (
    prototype_roads_gdf[
        "knowledge_highway_class"
    ]
    .apply(road_hierarchy_score)
)

prototype_roads_gdf[
    "hierarchy_group"
] = (
    prototype_roads_gdf[
        "knowledge_highway_class"
    ]
    .apply(road_hierarchy_group)
)

# ------------------------------------------------------------
# Critical infrastructure indicators
# ------------------------------------------------------------

prototype_roads_gdf["is_bridge_knowledge"] = (
    prototype_roads_gdf["is_bridge"]
    .apply(safe_boolean)
    if "is_bridge"
    in prototype_roads_gdf.columns
    else False
)

prototype_roads_gdf["is_tunnel_knowledge"] = (
    prototype_roads_gdf["is_tunnel"]
    .apply(safe_boolean)
    if "is_tunnel"
    in prototype_roads_gdf.columns
    else False
)

prototype_roads_gdf["is_oneway_knowledge"] = (
    prototype_roads_gdf["is_oneway"]
    .apply(safe_boolean)
    if "is_oneway"
    in prototype_roads_gdf.columns
    else False
)

prototype_roads_gdf["is_major_road"] = (
    prototype_roads_gdf[
        "hierarchy_score"
    ]
    >= 3
)

prototype_roads_gdf["is_high_capacity_road"] = (
    prototype_roads_gdf[
        "hierarchy_score"
    ]
    >= 5
)

prototype_roads_gdf["is_critical_link"] = (
    prototype_roads_gdf[
        "is_bridge_knowledge"
    ]
    |
    prototype_roads_gdf[
        "is_tunnel_knowledge"
    ]
    |
    prototype_roads_gdf[
        "is_high_capacity_road"
    ]
)

# ------------------------------------------------------------
# Numeric cleanup
# ------------------------------------------------------------

for column in [
    "length_m",
    "lanes_numeric",
    "maxspeed_numeric",
]:
    if column in prototype_roads_gdf.columns:
        prototype_roads_gdf[column] = (
            pd.to_numeric(
                prototype_roads_gdf[column],
                errors="coerce",
            )
        )

# ------------------------------------------------------------
# Summaries
# ------------------------------------------------------------

print("\nROADWAY CLASS DISTRIBUTION")
print("-" * 72)

print(
    prototype_roads_gdf[
        "knowledge_highway_class"
    ]
    .value_counts()
    .to_string()
)

print("\nROADWAY HIERARCHY GROUP DISTRIBUTION")
print("-" * 72)

print(
    prototype_roads_gdf[
        "hierarchy_group"
    ]
    .value_counts()
    .to_string()
)

print("\nCRITICAL INFRASTRUCTURE COUNTS")
print("-" * 72)

print(
    f"Major roadway edges       : "
    f"{prototype_roads_gdf['is_major_road'].sum():,}"
)

print(
    f"High-capacity roadway edges: "
    f"{prototype_roads_gdf['is_high_capacity_road'].sum():,}"
)

print(
    f"Bridge edges              : "
    f"{prototype_roads_gdf['is_bridge_knowledge'].sum():,}"
)

print(
    f"Tunnel edges              : "
    f"{prototype_roads_gdf['is_tunnel_knowledge'].sum():,}"
)

print(
    f"Critical-link edges       : "
    f"{prototype_roads_gdf['is_critical_link'].sum():,}"
)

print("\n" + "=" * 72)
print("ROAD HIERARCHY PREPARATION COMPLETE")
print("=" * 72)

ROAD HIERARCHY AND INFRASTRUCTURE PREPARATION

ROADWAY CLASS DISTRIBUTION
------------------------------------------------------------------------
knowledge_highway_class
residential      473
unclassified     236
tertiary         170
motorway_link     11
motorway           8
trunk              4

ROADWAY HIERARCHY GROUP DISTRIBUTION
------------------------------------------------------------------------
hierarchy_group
local              709
local_collector    170
limited_access      19
major_arterial       4

CRITICAL INFRASTRUCTURE COUNTS
------------------------------------------------------------------------
Major roadway edges       : 12
High-capacity roadway edges: 12
Bridge edges              : 33
Tunnel edges              : 4
Critical-link edges       : 42

ROAD HIERARCHY PREPARATION COMPLETE


In [6]:
# ============================================================
# CELL 5B: CORRECT ROADWAY-LINK HIERARCHY SCORES
# ============================================================

print("=" * 72)
print("ROADWAY-LINK HIERARCHY SCORE CORRECTION")
print("=" * 72)

# ------------------------------------------------------------
# Complete hierarchy scoring dictionary
# ------------------------------------------------------------

ROAD_HIERARCHY_SCORE.update(
    {
        "motorway_link": 6,
        "trunk_link": 5,
        "primary_link": 4,
        "secondary_link": 3,
        "tertiary_link": 2,
    }
)

# Recalculate hierarchy attributes.
prototype_roads_gdf["hierarchy_score"] = (
    prototype_roads_gdf[
        "knowledge_highway_class"
    ]
    .apply(
        lambda road_class: ROAD_HIERARCHY_SCORE.get(
            road_class,
            0,
        )
    )
)

prototype_roads_gdf["is_major_road"] = (
    prototype_roads_gdf["hierarchy_score"] >= 3
)

prototype_roads_gdf["is_high_capacity_road"] = (
    prototype_roads_gdf["hierarchy_score"] >= 5
)

prototype_roads_gdf["is_critical_link"] = (
    prototype_roads_gdf["is_bridge_knowledge"]
    |
    prototype_roads_gdf["is_tunnel_knowledge"]
    |
    prototype_roads_gdf["is_high_capacity_road"]
)

print("\nUPDATED CRITICAL INFRASTRUCTURE COUNTS")
print("-" * 72)

print(
    f"Major roadway edges        : "
    f"{prototype_roads_gdf['is_major_road'].sum():,}"
)

print(
    f"High-capacity roadway edges: "
    f"{prototype_roads_gdf['is_high_capacity_road'].sum():,}"
)

print(
    f"Bridge edges               : "
    f"{prototype_roads_gdf['is_bridge_knowledge'].sum():,}"
)

print(
    f"Tunnel edges               : "
    f"{prototype_roads_gdf['is_tunnel_knowledge'].sum():,}"
)

print(
    f"Critical-link edges        : "
    f"{prototype_roads_gdf['is_critical_link'].sum():,}"
)

print("\nUPDATED HIERARCHY SCORE DISTRIBUTION")
print("-" * 72)

print(
    prototype_roads_gdf[
        [
            "knowledge_highway_class",
            "hierarchy_score",
        ]
    ]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\n" + "=" * 72)
print("ROADWAY-LINK HIERARCHY CORRECTION COMPLETE")
print("=" * 72)

ROADWAY-LINK HIERARCHY SCORE CORRECTION

UPDATED CRITICAL INFRASTRUCTURE COUNTS
------------------------------------------------------------------------
Major roadway edges        : 23
High-capacity roadway edges: 23
Bridge edges               : 33
Tunnel edges               : 4
Critical-link edges        : 51

UPDATED HIERARCHY SCORE DISTRIBUTION
------------------------------------------------------------------------
knowledge_highway_class  hierarchy_score
motorway                 6                    8
motorway_link            6                   11
residential              1                  473
tertiary                 2                  170
trunk                    5                    4
unclassified             1                  236

ROADWAY-LINK HIERARCHY CORRECTION COMPLETE


In [7]:
# ============================================================
# CELL 6: COMPUTE PROTOTYPE NODE CENTRALITY
# ============================================================

print("=" * 72)
print("PROTOTYPE NODE CENTRALITY ANALYSIS")
print("=" * 72)

# ------------------------------------------------------------
# Prepare an undirected graph
# ------------------------------------------------------------

prototype_undirected_graph = (
    ox.convert.to_undirected(
        prototype_graph
    )
)

print(
    f"\nUndirected graph nodes: "
    f"{prototype_undirected_graph.number_of_nodes():,}"
)

print(
    f"Undirected graph edges: "
    f"{prototype_undirected_graph.number_of_edges():,}"
)

# ------------------------------------------------------------
# Degree and degree centrality
# ------------------------------------------------------------

node_degree = dict(
    prototype_undirected_graph.degree()
)

degree_centrality = nx.degree_centrality(
    prototype_undirected_graph
)

# ------------------------------------------------------------
# Betweenness centrality
# ------------------------------------------------------------
# Edge length is used as the travel-distance weight.
# A higher value means the node lies on more shortest paths.

betweenness_centrality = nx.betweenness_centrality(
    prototype_undirected_graph,
    weight="length",
    normalized=True,
)

# ------------------------------------------------------------
# Closeness centrality
# ------------------------------------------------------------
# Convert length to a distance attribute explicitly.

for u, v, key, data in (
    prototype_undirected_graph.edges(
        keys=True,
        data=True,
    )
):
    edge_length = data.get("length", 1.0)

    try:
        edge_length = float(edge_length)
    except (TypeError, ValueError):
        edge_length = 1.0

    data["distance"] = max(
        edge_length,
        0.001,
    )

closeness_centrality = nx.closeness_centrality(
    prototype_undirected_graph,
    distance="distance",
)

# ------------------------------------------------------------
# Articulation points
# ------------------------------------------------------------
# Articulation points are nodes whose removal increases the
# number of connected components.

simple_graph = nx.Graph(
    prototype_undirected_graph
)

articulation_points = set(
    nx.articulation_points(
        simple_graph
    )
)

# ------------------------------------------------------------
# Add attributes to the node GeoDataFrame
# ------------------------------------------------------------

prototype_nodes_gdf = (
    prototype_nodes_gdf.copy()
)

# OSM node identifiers may be loaded as strings or integers.
prototype_nodes_gdf["node_id_string"] = (
    prototype_nodes_gdf["osmid"].astype(str)
    if "osmid" in prototype_nodes_gdf.columns
    else prototype_nodes_gdf.index.astype(str)
)

degree_lookup = {
    str(node): value
    for node, value in node_degree.items()
}

degree_centrality_lookup = {
    str(node): value
    for node, value in degree_centrality.items()
}

betweenness_lookup = {
    str(node): value
    for node, value in betweenness_centrality.items()
}

closeness_lookup = {
    str(node): value
    for node, value in closeness_centrality.items()
}

articulation_lookup = {
    str(node)
    for node in articulation_points
}

prototype_nodes_gdf["node_degree"] = (
    prototype_nodes_gdf[
        "node_id_string"
    ]
    .map(degree_lookup)
    .fillna(0)
    .astype(int)
)

prototype_nodes_gdf["degree_centrality"] = (
    prototype_nodes_gdf[
        "node_id_string"
    ]
    .map(degree_centrality_lookup)
    .fillna(0.0)
)

prototype_nodes_gdf["betweenness_centrality"] = (
    prototype_nodes_gdf[
        "node_id_string"
    ]
    .map(betweenness_lookup)
    .fillna(0.0)
)

prototype_nodes_gdf["closeness_centrality"] = (
    prototype_nodes_gdf[
        "node_id_string"
    ]
    .map(closeness_lookup)
    .fillna(0.0)
)

prototype_nodes_gdf["is_articulation_point"] = (
    prototype_nodes_gdf[
        "node_id_string"
    ]
    .isin(articulation_lookup)
)

prototype_nodes_gdf["is_intersection"] = (
    prototype_nodes_gdf[
        "node_degree"
    ]
    >= 3
)

# ------------------------------------------------------------
# Percentile ranks
# ------------------------------------------------------------

prototype_nodes_gdf[
    "betweenness_percentile"
] = (
    prototype_nodes_gdf[
        "betweenness_centrality"
    ]
    .rank(
        pct=True,
        method="average",
    )
)

prototype_nodes_gdf[
    "closeness_percentile"
] = (
    prototype_nodes_gdf[
        "closeness_centrality"
    ]
    .rank(
        pct=True,
        method="average",
    )
)

prototype_nodes_gdf[
    "degree_percentile"
] = (
    prototype_nodes_gdf[
        "node_degree"
    ]
    .rank(
        pct=True,
        method="average",
    )
)

# ------------------------------------------------------------
# Composite node-importance score
# ------------------------------------------------------------

prototype_nodes_gdf[
    "node_importance_score"
] = (
    0.50
    * prototype_nodes_gdf[
        "betweenness_percentile"
    ]
    +
    0.25
    * prototype_nodes_gdf[
        "closeness_percentile"
    ]
    +
    0.25
    * prototype_nodes_gdf[
        "degree_percentile"
    ]
)

prototype_nodes_gdf[
    "critical_node"
] = (
    (
        prototype_nodes_gdf[
            "node_importance_score"
        ]
        >= 0.80
    )
    |
    prototype_nodes_gdf[
        "is_articulation_point"
    ]
)

# ------------------------------------------------------------
# Summaries
# ------------------------------------------------------------

print("\nNODE CENTRALITY SUMMARY")
print("-" * 72)

centrality_columns = [
    "node_degree",
    "degree_centrality",
    "betweenness_centrality",
    "closeness_centrality",
    "node_importance_score",
]

print(
    prototype_nodes_gdf[
        centrality_columns
    ]
    .describe()
    .round(6)
    .to_string()
)

print("\nCRITICAL NODE COUNTS")
print("-" * 72)

print(
    f"Intersections       : "
    f"{prototype_nodes_gdf['is_intersection'].sum():,}"
)

print(
    f"Articulation points : "
    f"{prototype_nodes_gdf['is_articulation_point'].sum():,}"
)

print(
    f"Critical nodes      : "
    f"{prototype_nodes_gdf['critical_node'].sum():,}"
)

print("\nTOP 15 NODES BY IMPORTANCE")
print("-" * 72)

top_node_columns = [
    "node_id_string",
    "node_degree",
    "betweenness_centrality",
    "closeness_centrality",
    "node_importance_score",
    "is_intersection",
    "is_articulation_point",
    "critical_node",
]

print(
    prototype_nodes_gdf[
        top_node_columns
    ]
    .sort_values(
        "node_importance_score",
        ascending=False,
    )
    .head(15)
    .round(6)
    .to_string(index=False)
)

print("\n" + "=" * 72)
print("PROTOTYPE NODE CENTRALITY COMPLETE")
print("=" * 72)

PROTOTYPE NODE CENTRALITY ANALYSIS



Undirected graph nodes: 449
Undirected graph edges: 542



NODE CENTRALITY SUMMARY
------------------------------------------------------------------------
       node_degree  degree_centrality  betweenness_centrality  closeness_centrality  node_importance_score
count   449.000000         449.000000              449.000000            449.000000             449.000000
mean      2.414254           0.005389                0.040636              0.000188               0.501114
std       1.003286           0.002239                0.069112              0.000061               0.241306
min       1.000000           0.002232                0.000000              0.000004               0.122494
25%       1.000000           0.002232                0.000000              0.000155               0.257517
50%       3.000000           0.006696                0.008339              0.000198               0.535913
75%       3.000000           0.006696                0.052233              0.000222               0.685134
max       4.000000           0.008929         

In [8]:
# ============================================================
# CELL 7: TRANSFER NODE IMPORTANCE TO ROAD EDGES
# ============================================================

print("=" * 72)
print("EDGE-LEVEL TOPOLOGICAL IMPORTANCE")
print("=" * 72)

prototype_roads_gdf = (
    prototype_roads_gdf.copy()
)

# ------------------------------------------------------------
# Identify edge endpoint fields
# ------------------------------------------------------------

if "u" not in prototype_roads_gdf.columns:
    prototype_roads_gdf["u"] = (
        prototype_roads_gdf.index.get_level_values(0)
        if isinstance(
            prototype_roads_gdf.index,
            pd.MultiIndex,
        )
        else np.nan
    )

if "v" not in prototype_roads_gdf.columns:
    prototype_roads_gdf["v"] = (
        prototype_roads_gdf.index.get_level_values(1)
        if isinstance(
            prototype_roads_gdf.index,
            pd.MultiIndex,
        )
        else np.nan
    )

prototype_roads_gdf["u_string"] = (
    prototype_roads_gdf["u"].astype(str)
)

prototype_roads_gdf["v_string"] = (
    prototype_roads_gdf["v"].astype(str)
)

node_importance_lookup = (
    prototype_nodes_gdf
    .set_index("node_id_string")[
        "node_importance_score"
    ]
    .to_dict()
)

node_betweenness_lookup = (
    prototype_nodes_gdf
    .set_index("node_id_string")[
        "betweenness_centrality"
    ]
    .to_dict()
)

critical_node_lookup = (
    prototype_nodes_gdf
    .set_index("node_id_string")[
        "critical_node"
    ]
    .to_dict()
)

articulation_lookup = (
    prototype_nodes_gdf
    .set_index("node_id_string")[
        "is_articulation_point"
    ]
    .to_dict()
)

# ------------------------------------------------------------
# Add endpoint-node features
# ------------------------------------------------------------

prototype_roads_gdf["u_node_importance"] = (
    prototype_roads_gdf[
        "u_string"
    ]
    .map(node_importance_lookup)
    .fillna(0.0)
)

prototype_roads_gdf["v_node_importance"] = (
    prototype_roads_gdf[
        "v_string"
    ]
    .map(node_importance_lookup)
    .fillna(0.0)
)

prototype_roads_gdf[
    "mean_endpoint_importance"
] = (
    prototype_roads_gdf[
        [
            "u_node_importance",
            "v_node_importance",
        ]
    ]
    .mean(axis=1)
)

prototype_roads_gdf[
    "max_endpoint_importance"
] = (
    prototype_roads_gdf[
        [
            "u_node_importance",
            "v_node_importance",
        ]
    ]
    .max(axis=1)
)

prototype_roads_gdf["u_betweenness"] = (
    prototype_roads_gdf[
        "u_string"
    ]
    .map(node_betweenness_lookup)
    .fillna(0.0)
)

prototype_roads_gdf["v_betweenness"] = (
    prototype_roads_gdf[
        "v_string"
    ]
    .map(node_betweenness_lookup)
    .fillna(0.0)
)

prototype_roads_gdf[
    "mean_endpoint_betweenness"
] = (
    prototype_roads_gdf[
        [
            "u_betweenness",
            "v_betweenness",
        ]
    ]
    .mean(axis=1)
)

prototype_roads_gdf[
    "touches_critical_node"
] = (
    prototype_roads_gdf[
        "u_string"
    ]
    .map(critical_node_lookup)
    .fillna(False)
    |
    prototype_roads_gdf[
        "v_string"
    ]
    .map(critical_node_lookup)
    .fillna(False)
)

prototype_roads_gdf[
    "touches_articulation_point"
] = (
    prototype_roads_gdf[
        "u_string"
    ]
    .map(articulation_lookup)
    .fillna(False)
    |
    prototype_roads_gdf[
        "v_string"
    ]
    .map(articulation_lookup)
    .fillna(False)
)

# ------------------------------------------------------------
# Edge criticality score
# ------------------------------------------------------------

prototype_roads_gdf[
    "edge_criticality_score"
] = (
    0.40
    * (
        prototype_roads_gdf[
            "hierarchy_score"
        ]
        / max(
            ROAD_HIERARCHY_SCORE.values()
        )
    )
    +
    0.35
    * prototype_roads_gdf[
        "max_endpoint_importance"
    ]
    +
    0.15
    * prototype_roads_gdf[
        "is_bridge_knowledge"
    ].astype(float)
    +
    0.05
    * prototype_roads_gdf[
        "is_tunnel_knowledge"
    ].astype(float)
    +
    0.05
    * prototype_roads_gdf[
        "touches_articulation_point"
    ].astype(float)
)

prototype_roads_gdf[
    "edge_criticality_percentile"
] = (
    prototype_roads_gdf[
        "edge_criticality_score"
    ]
    .rank(
        pct=True,
        method="average",
    )
)

prototype_roads_gdf[
    "critical_transport_edge"
] = (
    prototype_roads_gdf[
        "edge_criticality_percentile"
    ]
    >= 0.80
)

print("\nEDGE CRITICALITY SUMMARY")
print("-" * 72)

print(
    prototype_roads_gdf[
        [
            "hierarchy_score",
            "mean_endpoint_importance",
            "max_endpoint_importance",
            "edge_criticality_score",
        ]
    ]
    .describe()
    .round(6)
    .to_string()
)

print("\nEDGE IMPORTANCE COUNTS")
print("-" * 72)

print(
    f"Edges touching critical nodes     : "
    f"{prototype_roads_gdf['touches_critical_node'].sum():,}"
)

print(
    f"Edges touching articulation points: "
    f"{prototype_roads_gdf['touches_articulation_point'].sum():,}"
)

print(
    f"Critical transportation edges     : "
    f"{prototype_roads_gdf['critical_transport_edge'].sum():,}"
)

print("\nTOP 20 CRITICAL TRANSPORTATION EDGES")
print("-" * 72)

edge_display_columns = [
    "knowledge_highway_class",
    "hierarchy_group",
    "length_m",
    "is_bridge_knowledge",
    "is_tunnel_knowledge",
    "max_endpoint_importance",
    "touches_articulation_point",
    "edge_criticality_score",
    "critical_transport_edge",
]

print(
    prototype_roads_gdf[
        edge_display_columns
    ]
    .sort_values(
        "edge_criticality_score",
        ascending=False,
    )
    .head(20)
    .round(6)
    .to_string(index=False)
)

print("\n" + "=" * 72)
print("EDGE-LEVEL TOPOLOGICAL IMPORTANCE COMPLETE")
print("=" * 72)

EDGE-LEVEL TOPOLOGICAL IMPORTANCE

EDGE CRITICALITY SUMMARY
------------------------------------------------------------------------
       hierarchy_score  mean_endpoint_importance  max_endpoint_importance  edge_criticality_score
count       902.000000                902.000000               902.000000              902.000000
mean          1.311530                  0.576119                 0.654047                0.355986
std           0.830509                  0.184830                 0.164730                0.089966
min           1.000000                  0.122494                 0.122494                0.109540
25%           1.000000                  0.433463                 0.546492                0.305002
50%           1.000000                  0.587138                 0.657572                0.346803
75%           1.000000                  0.716662                 0.793708                0.397764
max           6.000000                  0.949332                 0.990535          

In [9]:
# ============================================================
# CELL 8: PROTOTYPE CONNECTIVITY AND REDUNDANCY ANALYSIS
# ============================================================

print("=" * 72)
print("PROTOTYPE NETWORK CONNECTIVITY AND REDUNDANCY")
print("=" * 72)

# ------------------------------------------------------------
# 1. Prepare a simple undirected graph
# ------------------------------------------------------------

simple_graph = nx.Graph()

for node, node_data in prototype_undirected_graph.nodes(
    data=True
):
    simple_graph.add_node(
        node,
        **node_data,
    )

for u, v, edge_data in prototype_undirected_graph.edges(
    data=True
):
    edge_length = edge_data.get(
        "length",
        1.0,
    )

    try:
        edge_length = float(edge_length)
    except (TypeError, ValueError):
        edge_length = 1.0

    edge_length = max(
        edge_length,
        0.001,
    )

    if simple_graph.has_edge(u, v):
        existing_length = simple_graph[u][v].get(
            "length",
            edge_length,
        )

        simple_graph[u][v]["length"] = min(
            existing_length,
            edge_length,
        )

    else:
        simple_graph.add_edge(
            u,
            v,
            length=edge_length,
        )

print(
    f"\nSimple graph nodes : "
    f"{simple_graph.number_of_nodes():,}"
)

print(
    f"Simple graph edges : "
    f"{simple_graph.number_of_edges():,}"
)

# ------------------------------------------------------------
# 2. Connected components
# ------------------------------------------------------------

connected_components = list(
    nx.connected_components(
        simple_graph
    )
)

connected_components = sorted(
    connected_components,
    key=len,
    reverse=True,
)

component_sizes = [
    len(component)
    for component in connected_components
]

largest_component_nodes = (
    connected_components[0]
    if connected_components
    else set()
)

largest_component_graph = (
    simple_graph.subgraph(
        largest_component_nodes
    )
    .copy()
)

component_lookup = {}

for component_id, component_nodes in enumerate(
    connected_components,
    start=1,
):
    for node in component_nodes:
        component_lookup[str(node)] = component_id

prototype_nodes_gdf[
    "connected_component_id"
] = (
    prototype_nodes_gdf[
        "node_id_string"
    ]
    .map(component_lookup)
)

prototype_nodes_gdf[
    "in_largest_component"
] = (
    prototype_nodes_gdf[
        "connected_component_id"
    ]
    == 1
)

largest_component_share = (
    len(largest_component_nodes)
    / simple_graph.number_of_nodes()
    if simple_graph.number_of_nodes() > 0
    else 0.0
)

# ------------------------------------------------------------
# 3. Articulation points and bridge edges
# ------------------------------------------------------------

network_articulation_points = set(
    nx.articulation_points(
        simple_graph
    )
)

network_bridge_edges = set(
    nx.bridges(
        simple_graph
    )
)

# Store both orientations for lookup.
network_bridge_lookup = set()

for u, v in network_bridge_edges:
    network_bridge_lookup.add(
        (
            str(u),
            str(v),
        )
    )

    network_bridge_lookup.add(
        (
            str(v),
            str(u),
        )
    )

prototype_nodes_gdf[
    "is_network_articulation"
] = (
    prototype_nodes_gdf[
        "node_id_string"
    ]
    .isin(
        {
            str(node)
            for node in network_articulation_points
        }
    )
)

prototype_roads_gdf[
    "is_network_bridge_edge"
] = [
    (
        str(u),
        str(v),
    )
    in network_bridge_lookup
    for u, v in zip(
        prototype_roads_gdf["u"],
        prototype_roads_gdf["v"],
    )
]

# ------------------------------------------------------------
# 4. Cycle participation
# ------------------------------------------------------------
# Roads in cycles generally have more routing alternatives.
# Roads outside cycles are more likely to be single-access links.

cycle_basis = nx.cycle_basis(
    simple_graph
)

cycle_edge_lookup = set()

for cycle in cycle_basis:

    if len(cycle) < 3:
        continue

    cycle_pairs = list(
        zip(
            cycle,
            cycle[1:] + [cycle[0]],
        )
    )

    for u, v in cycle_pairs:
        cycle_edge_lookup.add(
            (
                str(u),
                str(v),
            )
        )

        cycle_edge_lookup.add(
            (
                str(v),
                str(u),
            )
        )

prototype_roads_gdf[
    "participates_in_cycle"
] = [
    (
        str(u),
        str(v),
    )
    in cycle_edge_lookup
    for u, v in zip(
        prototype_roads_gdf["u"],
        prototype_roads_gdf["v"],
    )
]

prototype_roads_gdf[
    "limited_route_redundancy"
] = (
    ~prototype_roads_gdf[
        "participates_in_cycle"
    ]
)

# ------------------------------------------------------------
# 5. Local edge connectivity
# ------------------------------------------------------------
# Measures the minimum number of road edges that must be
# removed to disconnect the endpoints of each road segment.
#
# Interpretation:
#   1 = no alternative edge-disjoint route
#   2 = one alternative route
#   3 = multiple alternatives
#   4 = four or more alternatives (capped for efficiency)

from networkx.algorithms.connectivity import (
    local_edge_connectivity,
)

unique_endpoint_pairs = (
    prototype_roads_gdf[
        ["u_string", "v_string"]
    ]
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)

graph_node_lookup = {
    str(node): node
    for node in simple_graph.nodes
}

local_connectivity_lookup = {}

total_pairs = len(unique_endpoint_pairs)

print(
    f"\nComputing local edge connectivity for "
    f"{total_pairs:,} unique road endpoint pairs..."
)

for pair_index, endpoint_row in unique_endpoint_pairs.iterrows():

    u_string = endpoint_row["u_string"]
    v_string = endpoint_row["v_string"]

    u_node = graph_node_lookup.get(u_string)
    v_node = graph_node_lookup.get(v_string)

    if (
        u_node is None
        or v_node is None
        or u_node == v_node
    ):
        connectivity_value = 0

    else:
        try:
            connectivity_value = local_edge_connectivity(
                simple_graph,
                u_node,
                v_node,
                cutoff=4,
            )

        except (
            nx.NetworkXError,
            nx.NodeNotFound,
            nx.NetworkXNoPath,
            ValueError,
        ):
            connectivity_value = 0

    connectivity_value = int(connectivity_value)

    local_connectivity_lookup[
        (u_string, v_string)
    ] = connectivity_value

    local_connectivity_lookup[
        (v_string, u_string)
    ] = connectivity_value

    if (
        (pair_index + 1) % 100 == 0
        or pair_index + 1 == total_pairs
    ):
        print(
            f"Processed "
            f"{pair_index + 1:,}/"
            f"{total_pairs:,} endpoint pairs"
        )

prototype_roads_gdf[
    "local_edge_connectivity"
] = [
    local_connectivity_lookup.get(
        (str(u), str(v)),
        0,
    )
    for u, v in zip(
        prototype_roads_gdf["u"],
        prototype_roads_gdf["v"],
    )
]

print("\nLOCAL EDGE CONNECTIVITY DISTRIBUTION")
print("-" * 72)

print(
    prototype_roads_gdf[
        "local_edge_connectivity"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)
# ------------------------------------------------------------
# 6. Edge redundancy category
# ------------------------------------------------------------

def classify_redundancy(row):
    """
    Assign a qualitative route-redundancy category.
    """

    if row["is_network_bridge_edge"]:
        return "none"

    if row["local_edge_connectivity"] <= 1:
        return "low"

    if row["local_edge_connectivity"] == 2:
        return "moderate"

    return "high"


prototype_roads_gdf[
    "route_redundancy_class"
] = (
    prototype_roads_gdf.apply(
        classify_redundancy,
        axis=1,
    )
)

# ------------------------------------------------------------
# 7. Vulnerability indicators
# ------------------------------------------------------------

prototype_roads_gdf[
    "single_access_link"
] = (
    prototype_roads_gdf[
        "is_network_bridge_edge"
    ]
    |
    (
        prototype_roads_gdf[
            "local_edge_connectivity"
        ]
        <= 1
    )
)

prototype_roads_gdf[
    "critical_low_redundancy_edge"
] = (
    prototype_roads_gdf[
        "critical_transport_edge"
    ]
    &
    prototype_roads_gdf[
        "single_access_link"
    ]
)

prototype_roads_gdf[
    "bridge_bottleneck_edge"
] = (
    prototype_roads_gdf[
        "is_bridge_knowledge"
    ]
    &
    prototype_roads_gdf[
        "single_access_link"
    ]
)

prototype_roads_gdf[
    "major_road_bottleneck_edge"
] = (
    prototype_roads_gdf[
        "is_major_road"
    ]
    &
    prototype_roads_gdf[
        "single_access_link"
    ]
)

# ------------------------------------------------------------
# 8. Redundancy score
# ------------------------------------------------------------

connectivity_max = max(
    prototype_roads_gdf[
        "local_edge_connectivity"
    ].max(),
    1,
)

prototype_roads_gdf[
    "normalized_local_connectivity"
] = (
    prototype_roads_gdf[
        "local_edge_connectivity"
    ]
    / connectivity_max
)

prototype_roads_gdf[
    "route_redundancy_score"
] = (
    0.60
    * prototype_roads_gdf[
        "normalized_local_connectivity"
    ]
    +
    0.40
    * prototype_roads_gdf[
        "participates_in_cycle"
    ].astype(float)
)

prototype_roads_gdf[
    "topological_vulnerability_score"
] = (
    0.45
    * prototype_roads_gdf[
        "edge_criticality_score"
    ]
    +
    0.35
    * (
        1
        - prototype_roads_gdf[
            "route_redundancy_score"
        ]
    )
    +
    0.20
    * prototype_roads_gdf[
        "is_network_bridge_edge"
    ].astype(float)
)

prototype_roads_gdf[
    "topological_vulnerability_percentile"
] = (
    prototype_roads_gdf[
        "topological_vulnerability_score"
    ]
    .rank(
        pct=True,
        method="average",
    )
)

prototype_roads_gdf[
    "high_topological_vulnerability"
] = (
    prototype_roads_gdf[
        "topological_vulnerability_percentile"
    ]
    >= 0.80
)

# ------------------------------------------------------------
# 9. Network-level summary
# ------------------------------------------------------------

network_summary = {
    "scene_id": PROTOTYPE_SCENE_ID,
    "node_count": simple_graph.number_of_nodes(),
    "edge_count": simple_graph.number_of_edges(),
    "connected_component_count": len(
        connected_components
    ),
    "largest_component_nodes": len(
        largest_component_nodes
    ),
    "largest_component_share": (
        largest_component_share
    ),
    "articulation_point_count": len(
        network_articulation_points
    ),
    "network_bridge_edge_count": len(
        network_bridge_edges
    ),
    "cycle_count": len(
        cycle_basis
    ),
    "edges_in_cycles": int(
        prototype_roads_gdf[
            "participates_in_cycle"
        ].sum()
    ),
    "single_access_edge_count": int(
        prototype_roads_gdf[
            "single_access_link"
        ].sum()
    ),
    "critical_low_redundancy_count": int(
        prototype_roads_gdf[
            "critical_low_redundancy_edge"
        ].sum()
    ),
    "bridge_bottleneck_count": int(
        prototype_roads_gdf[
            "bridge_bottleneck_edge"
        ].sum()
    ),
    "major_road_bottleneck_count": int(
        prototype_roads_gdf[
            "major_road_bottleneck_edge"
        ].sum()
    ),
    "mean_local_edge_connectivity": float(
        prototype_roads_gdf[
            "local_edge_connectivity"
        ].mean()
    ),
    "mean_route_redundancy_score": float(
        prototype_roads_gdf[
            "route_redundancy_score"
        ].mean()
    ),
}

prototype_network_summary_df = pd.DataFrame(
    [network_summary]
)

# ------------------------------------------------------------
# 10. Console summaries
# ------------------------------------------------------------

print("\nCONNECTED COMPONENT SUMMARY")
print("-" * 72)

print(
    f"Connected components       : "
    f"{len(connected_components):,}"
)

print(
    f"Largest component nodes    : "
    f"{len(largest_component_nodes):,}"
)

print(
    f"Largest component share    : "
    f"{largest_component_share:.3f}"
)

print(
    f"Component sizes             : "
    f"{component_sizes}"
)

print("\nNETWORK BOTTLENECK SUMMARY")
print("-" * 72)

print(
    f"Articulation points         : "
    f"{len(network_articulation_points):,}"
)

print(
    f"Network bridge edges        : "
    f"{len(network_bridge_edges):,}"
)

print(
    f"Independent cycles          : "
    f"{len(cycle_basis):,}"
)

print(
    f"Road records in cycles      : "
    f"{prototype_roads_gdf['participates_in_cycle'].sum():,}"
)

print(
    f"Single-access road records  : "
    f"{prototype_roads_gdf['single_access_link'].sum():,}"
)

print("\nCRITICAL VULNERABILITY COUNTS")
print("-" * 72)

print(
    f"Critical low-redundancy edges: "
    f"{prototype_roads_gdf['critical_low_redundancy_edge'].sum():,}"
)

print(
    f"Bridge bottleneck edges       : "
    f"{prototype_roads_gdf['bridge_bottleneck_edge'].sum():,}"
)

print(
    f"Major-road bottleneck edges   : "
    f"{prototype_roads_gdf['major_road_bottleneck_edge'].sum():,}"
)

print(
    f"High-vulnerability edges      : "
    f"{prototype_roads_gdf['high_topological_vulnerability'].sum():,}"
)

print("\nROUTE REDUNDANCY DISTRIBUTION")
print("-" * 72)

print(
    prototype_roads_gdf[
        "route_redundancy_class"
    ]
    .value_counts()
    .to_string()
)

print("\nTOP 20 EDGES BY TOPOLOGICAL VULNERABILITY")
print("-" * 72)

vulnerability_display_columns = [
    "knowledge_highway_class",
    "hierarchy_group",
    "length_m",
    "is_bridge_knowledge",
    "is_network_bridge_edge",
    "local_edge_connectivity",
    "participates_in_cycle",
    "route_redundancy_class",
    "edge_criticality_score",
    "topological_vulnerability_score",
    "high_topological_vulnerability",
]

print(
    prototype_roads_gdf[
        vulnerability_display_columns
    ]
    .sort_values(
        "topological_vulnerability_score",
        ascending=False,
    )
    .head(20)
    .round(6)
    .to_string(index=False)
)

print("\n" + "=" * 72)
print("PROTOTYPE CONNECTIVITY AND REDUNDANCY COMPLETE")
print("=" * 72)

PROTOTYPE NETWORK CONNECTIVITY AND REDUNDANCY

Simple graph nodes : 449
Simple graph edges : 542

Computing local edge connectivity for 902 unique road endpoint pairs...


Processed 100/902 endpoint pairs


Processed 200/902 endpoint pairs


Processed 300/902 endpoint pairs


Processed 400/902 endpoint pairs


Processed 500/902 endpoint pairs


Processed 600/902 endpoint pairs


Processed 700/902 endpoint pairs


Processed 800/902 endpoint pairs


Processed 900/902 endpoint pairs
Processed 902/902 endpoint pairs

LOCAL EDGE CONNECTIVITY DISTRIBUTION
------------------------------------------------------------------------
local_edge_connectivity
1    393
2    255
3    225
4     29

CONNECTED COMPONENT SUMMARY
------------------------------------------------------------------------
Connected components       : 9
Largest component nodes    : 420
Largest component share    : 0.935
Component sizes             : [420, 14, 3, 2, 2, 2, 2, 2, 2]

NETWORK BOTTLENECK SUMMARY
------------------------------------------------------------------------
Articulation points         : 149
Network bridge edges        : 200
Independent cycles          : 102
Road records in cycles      : 509
Single-access road records  : 393

CRITICAL VULNERABILITY COUNTS
------------------------------------------------------------------------
Critical low-redundancy edges: 54
Bridge bottleneck edges       : 21
Major-road bottleneck edges   : 10
High-vulnerability edg

In [10]:
# ============================================================
# CELL 8B: VALIDATE REDUNDANCY AND BOTTLENECK RELATIONSHIPS
# ============================================================

print("=" * 72)
print("REDUNDANCY AND BOTTLENECK VALIDATION")
print("=" * 72)

validation_summary = pd.DataFrame(
    {
        "metric": [
            "Directed road records",
            "Simple undirected edges",
            "Local connectivity = 1",
            "Network bridge road records",
            "Outside cycle road records",
            "Single-access road records",
            "Physical bridge road records",
            "Physical bridge bottlenecks",
            "Critical low-redundancy records",
        ],
        "count": [
            len(prototype_roads_gdf),
            simple_graph.number_of_edges(),
            int(
                (
                    prototype_roads_gdf[
                        "local_edge_connectivity"
                    ]
                    == 1
                ).sum()
            ),
            int(
                prototype_roads_gdf[
                    "is_network_bridge_edge"
                ].sum()
            ),
            int(
                (
                    ~prototype_roads_gdf[
                        "participates_in_cycle"
                    ]
                ).sum()
            ),
            int(
                prototype_roads_gdf[
                    "single_access_link"
                ].sum()
            ),
            int(
                prototype_roads_gdf[
                    "is_bridge_knowledge"
                ].sum()
            ),
            int(
                prototype_roads_gdf[
                    "bridge_bottleneck_edge"
                ].sum()
            ),
            int(
                prototype_roads_gdf[
                    "critical_low_redundancy_edge"
                ].sum()
            ),
        ],
    }
)

print("\nVALIDATION SUMMARY")
print("-" * 72)

print(
    validation_summary.to_string(
        index=False
    )
)

# ------------------------------------------------------------
# Cross-tabulate connectivity and cycle participation
# ------------------------------------------------------------

connectivity_cycle_table = pd.crosstab(
    prototype_roads_gdf[
        "local_edge_connectivity"
    ],
    prototype_roads_gdf[
        "participates_in_cycle"
    ],
    margins=True,
)

connectivity_cycle_table.columns = [
    "outside_cycle"
    if column is False
    else "inside_cycle"
    if column is True
    else str(column)
    for column in connectivity_cycle_table.columns
]

print("\nLOCAL CONNECTIVITY × CYCLE PARTICIPATION")
print("-" * 72)

print(
    connectivity_cycle_table.to_string()
)

# ------------------------------------------------------------
# Cross-tabulate physical and topological bridges
# ------------------------------------------------------------

physical_topological_bridge_table = pd.crosstab(
    prototype_roads_gdf[
        "is_bridge_knowledge"
    ],
    prototype_roads_gdf[
        "is_network_bridge_edge"
    ],
    margins=True,
)

physical_topological_bridge_table.index = [
    "not_physical_bridge"
    if value is False
    else "physical_bridge"
    if value is True
    else str(value)
    for value in physical_topological_bridge_table.index
]

physical_topological_bridge_table.columns = [
    "not_graph_bridge"
    if value is False
    else "graph_bridge"
    if value is True
    else str(value)
    for value in physical_topological_bridge_table.columns
]

print("\nPHYSICAL BRIDGE × GRAPH BRIDGE")
print("-" * 72)

print(
    physical_topological_bridge_table.to_string()
)

# ------------------------------------------------------------
# Logical consistency checks
# ------------------------------------------------------------

consistency_checks = {
    "All graph bridges have local connectivity 1": bool(
        (
            prototype_roads_gdf.loc[
                prototype_roads_gdf[
                    "is_network_bridge_edge"
                ],
                "local_edge_connectivity",
            ]
            == 1
        ).all()
    ),
    "All graph bridges are outside cycles": bool(
        (
            ~prototype_roads_gdf.loc[
                prototype_roads_gdf[
                    "is_network_bridge_edge"
                ],
                "participates_in_cycle",
            ]
        ).all()
    ),
    "All bridge bottlenecks are physical bridges": bool(
        (
            prototype_roads_gdf.loc[
                prototype_roads_gdf[
                    "bridge_bottleneck_edge"
                ],
                "is_bridge_knowledge",
            ]
        ).all()
    ),
    "All critical low-redundancy edges are critical": bool(
        (
            prototype_roads_gdf.loc[
                prototype_roads_gdf[
                    "critical_low_redundancy_edge"
                ],
                "critical_transport_edge",
            ]
        ).all()
    ),
}

print("\nLOGICAL CONSISTENCY CHECKS")
print("-" * 72)

for check_name, check_result in consistency_checks.items():
    status = "PASS" if check_result else "REVIEW"

    print(
        f"{status:>6} | {check_name}"
    )

print("\n" + "=" * 72)
print("REDUNDANCY VALIDATION COMPLETE")
print("=" * 72)

REDUNDANCY AND BOTTLENECK VALIDATION

VALIDATION SUMMARY
------------------------------------------------------------------------
                         metric  count
          Directed road records    902
        Simple undirected edges    542
         Local connectivity = 1    393
    Network bridge road records    393
     Outside cycle road records    393
     Single-access road records    393
   Physical bridge road records     33
    Physical bridge bottlenecks     21
Critical low-redundancy records     54

LOCAL CONNECTIVITY × CYCLE PARTICIPATION
------------------------------------------------------------------------
                         outside_cycle  inside_cycle  All
local_edge_connectivity                                  
1                                  393             0  393
2                                    0           255  255
3                                    0           225  225
4                                    0            29   29
All              

In [11]:
# ============================================================
# CELL 9: BUILD TRANSPORTATION KNOWLEDGE OBJECTS
# ============================================================

print("=" * 72)
print("BUILD TRANSPORTATION KNOWLEDGE OBJECTS")
print("=" * 72)


def hierarchy_label(score):

    if score >= 6:
        return "Motorway"

    if score >= 5:
        return "Trunk"

    if score >= 4:
        return "Primary"

    if score >= 3:
        return "Secondary"

    if score >= 2:
        return "Tertiary"

    return "Local"


def redundancy_label(connectivity):

    if connectivity <= 1:
        return "None"

    if connectivity == 2:
        return "Low"

    if connectivity == 3:
        return "Moderate"

    return "High"


def vulnerability_label(score):

    if score >= 0.80:
        return "Very High"

    if score >= 0.60:
        return "High"

    if score >= 0.40:
        return "Moderate"

    if score >= 0.20:
        return "Low"

    return "Very Low"


def network_role(row):

    if row["bridge_bottleneck_edge"]:
        return "Critical Bridge"

    if row["critical_low_redundancy_edge"]:
        return "Critical Corridor"

    if row["critical_transport_edge"]:
        return "Primary Connector"

    if row["is_network_bridge_edge"]:
        return "Single Access"

    return "Local Access"


prototype_roads_gdf["HierarchyLabel"] = (
    prototype_roads_gdf["hierarchy_score"]
    .apply(hierarchy_label)
)

prototype_roads_gdf["RedundancyLabel"] = (
    prototype_roads_gdf["local_edge_connectivity"]
    .apply(redundancy_label)
)

prototype_roads_gdf["VulnerabilityLabel"] = (
    prototype_roads_gdf[
        "topological_vulnerability_score"
    ].apply(vulnerability_label)
)

prototype_roads_gdf["NetworkRole"] = (
    prototype_roads_gdf.apply(
        network_role,
        axis=1,
    )
)

print()

print(
    prototype_roads_gdf[
        [
            "HierarchyLabel",
            "RedundancyLabel",
            "VulnerabilityLabel",
            "NetworkRole",
        ]
    ]
    .head(20)
)

BUILD TRANSPORTATION KNOWLEDGE OBJECTS

   HierarchyLabel RedundancyLabel VulnerabilityLabel        NetworkRole
0        Motorway            None          Very High    Critical Bridge
1        Motorway            None          Very High    Critical Bridge
2        Motorway            None               High  Critical Corridor
3           Local            None               High      Single Access
4        Tertiary             Low                Low       Local Access
5        Tertiary        Moderate                Low  Primary Connector
6        Tertiary             Low                Low  Primary Connector
7           Local        Moderate                Low  Primary Connector
8           Local            None               High  Critical Corridor
9        Tertiary             Low                Low  Primary Connector
10       Tertiary             Low           Moderate  Primary Connector
11       Tertiary             Low                Low  Primary Connector
12       Tertiary       

In [12]:
# ============================================================
# CELL 10: GENERATE TRANSPORTATION KNOWLEDGE TOKENS
# ============================================================

print("=" * 72)
print("GENERATING TRANSPORTATION KNOWLEDGE TOKENS")
print("=" * 72)


def build_transport_token(row):

    tokens = []

    tokens.append(
        f"<Road:{row['HierarchyLabel']}>"
    )

    tokens.append(
        f"<Role:{row['NetworkRole']}>"
    )

    tokens.append(
        f"<Redundancy:{row['RedundancyLabel']}>"
    )

    tokens.append(
        f"<Vulnerability:{row['VulnerabilityLabel']}>"
    )

    if row["is_bridge_knowledge"]:
        tokens.append("<Bridge>")

    if row["is_tunnel_knowledge"]:
        tokens.append("<Tunnel>")

    if row["critical_transport_edge"]:
        tokens.append("<CriticalRoad>")

    if row["is_major_road"]:
        tokens.append("<MajorRoad>")

    return " ".join(tokens)


prototype_roads_gdf["TransportationToken"] = (
    prototype_roads_gdf.apply(
        build_transport_token,
        axis=1,
    )
)

print()

print(
    prototype_roads_gdf[
        [
            "TransportationToken"
        ]
    ]
    .head(20)
    .to_string(index=False)
)

GENERATING TRANSPORTATION KNOWLEDGE TOKENS

                                                                                                   TransportationToken
<Road:Motorway> <Role:Critical Bridge> <Redundancy:None> <Vulnerability:Very High> <Bridge> <CriticalRoad> <MajorRoad>
<Road:Motorway> <Role:Critical Bridge> <Redundancy:None> <Vulnerability:Very High> <Bridge> <CriticalRoad> <MajorRoad>
            <Road:Motorway> <Role:Critical Corridor> <Redundancy:None> <Vulnerability:High> <CriticalRoad> <MajorRoad>
                                              <Road:Local> <Role:Single Access> <Redundancy:None> <Vulnerability:High>
                                              <Road:Tertiary> <Role:Local Access> <Redundancy:Low> <Vulnerability:Low>
                     <Road:Tertiary> <Role:Primary Connector> <Redundancy:Moderate> <Vulnerability:Low> <CriticalRoad>
                          <Road:Tertiary> <Role:Primary Connector> <Redundancy:Low> <Vulnerability:Low> <CriticalRoad>
    

In [13]:
# ============================================================
# CELL 11: SCENE TRANSPORTATION PROFILE
# ============================================================

print("=" * 72)
print("SCENE TRANSPORTATION PROFILE")
print("=" * 72)

scene_profile = {

    "SceneID": PROTOTYPE_SCENE_ID,

    "RoadDensity":
        prototype_network_summary_df.iloc[0][
            "edge_count"
        ],

    "LargestComponent":
        prototype_network_summary_df.iloc[0][
            "largest_component_share"
        ],

    "BridgeCount":
        int(
            prototype_roads_gdf[
                "is_bridge_knowledge"
            ].sum()
        ),

    "CriticalRoads":
        int(
            prototype_roads_gdf[
                "critical_transport_edge"
            ].sum()
        ),

    "BridgeBottlenecks":
        int(
            prototype_roads_gdf[
                "bridge_bottleneck_edge"
            ].sum()
        ),

    "MajorRoads":
        int(
            prototype_roads_gdf[
                "is_major_road"
            ].sum()
        ),

    "Cycles":
        len(cycle_basis),

    "ArticulationPoints":
        len(network_articulation_points)

}

scene_profile_df = pd.DataFrame(
    [scene_profile]
)

print(scene_profile_df.T)

SCENE TRANSPORTATION PROFILE
                                0
SceneID             Spain_8565131
RoadDensity                   542
LargestComponent         0.935412
BridgeCount                    33
CriticalRoads                 180
BridgeBottlenecks              21
MajorRoads                     23
Cycles                        102
ArticulationPoints            149


In [14]:
# ============================================================
# CELL 10: GENERATE TRANSPORTATION KNOWLEDGE TOKENS
# ============================================================

print("=" * 72)
print("GENERATING TRANSPORTATION KNOWLEDGE TOKENS")
print("=" * 72)


def clean_token_value(value):
    """
    Convert a semantic label into a consistent token value.
    Example: 'Very High' -> 'VeryHigh'
    """
    return str(value).replace(" ", "").replace("-", "")


def build_transportation_token(row):
    """
    Build a semantic transportation representation for one
    road record using the labels produced in Cell 9.
    """

    tokens = [
        f"<RoadHierarchy:{clean_token_value(row['HierarchyLabel'])}>",
        f"<NetworkRole:{clean_token_value(row['NetworkRole'])}>",
        f"<RouteRedundancy:{clean_token_value(row['RedundancyLabel'])}>",
        f"<TopologicalVulnerability:{clean_token_value(row['VulnerabilityLabel'])}>",
    ]

    if bool(row["is_bridge_knowledge"]):
        tokens.append("<PhysicalBridge:Yes>")

    if bool(row["is_tunnel_knowledge"]):
        tokens.append("<Tunnel:Yes>")

    if bool(row["is_network_bridge_edge"]):
        tokens.append("<GraphBridge:Yes>")

    if bool(row["participates_in_cycle"]):
        tokens.append("<AlternateRouteStructure:Yes>")

    if bool(row["critical_transport_edge"]):
        tokens.append("<CriticalTransportEdge:Yes>")

    if bool(row["critical_low_redundancy_edge"]):
        tokens.append("<CriticalLowRedundancy:Yes>")

    if bool(row["bridge_bottleneck_edge"]):
        tokens.append("<BridgeBottleneck:Yes>")

    if bool(row["is_major_road"]):
        tokens.append("<MajorRoad:Yes>")

    return " ".join(tokens)


prototype_roads_gdf["TransportationToken"] = (
    prototype_roads_gdf.apply(
        build_transportation_token,
        axis=1,
    )
)

print("\nSAMPLE TRANSPORTATION KNOWLEDGE TOKENS")
print("-" * 72)

display_columns = [
    "knowledge_highway_class",
    "HierarchyLabel",
    "NetworkRole",
    "RedundancyLabel",
    "VulnerabilityLabel",
    "TransportationToken",
]

print(
    prototype_roads_gdf[
        display_columns
    ]
    .head(20)
    .to_string(index=False)
)

print("\nTOKEN GENERATION SUMMARY")
print("-" * 72)

print(
    f"Road records processed : "
    f"{len(prototype_roads_gdf):,}"
)

print(
    f"Tokens generated       : "
    f"{prototype_roads_gdf['TransportationToken'].notna().sum():,}"
)

print(
    f"Unique token patterns  : "
    f"{prototype_roads_gdf['TransportationToken'].nunique():,}"
)

print("\n" + "=" * 72)
print("TRANSPORTATION KNOWLEDGE TOKEN GENERATION COMPLETE")
print("=" * 72)

GENERATING TRANSPORTATION KNOWLEDGE TOKENS



SAMPLE TRANSPORTATION KNOWLEDGE TOKENS
------------------------------------------------------------------------
knowledge_highway_class HierarchyLabel       NetworkRole RedundancyLabel VulnerabilityLabel                                                                                                                                                                                                                                    TransportationToken
               motorway       Motorway   Critical Bridge            None          Very High <RoadHierarchy:Motorway> <NetworkRole:CriticalBridge> <RouteRedundancy:None> <TopologicalVulnerability:VeryHigh> <PhysicalBridge:Yes> <GraphBridge:Yes> <CriticalTransportEdge:Yes> <CriticalLowRedundancy:Yes> <BridgeBottleneck:Yes> <MajorRoad:Yes>
               motorway       Motorway   Critical Bridge            None          Very High <RoadHierarchy:Motorway> <NetworkRole:CriticalBridge> <RouteRedundancy:None> <TopologicalVulnerability:VeryHigh> <Phy

In [15]:
# ============================================================
# CELL 11: BUILD SCENE-LEVEL TRANSPORTATION PROFILE
# ============================================================

print("=" * 72)
print("BUILDING SCENE-LEVEL TRANSPORTATION PROFILE")
print("=" * 72)


def safe_share(count, total):
    """
    Return a proportion while avoiding division by zero.
    """
    if total == 0:
        return 0.0

    return float(count) / float(total)


total_road_records = len(prototype_roads_gdf)

total_nodes = simple_graph.number_of_nodes()
total_edges = simple_graph.number_of_edges()

largest_component_share = (
    len(max(nx.connected_components(simple_graph), key=len))
    / total_nodes
    if total_nodes > 0
    else 0.0
)

road_class_counts = (
    prototype_roads_gdf[
        "knowledge_highway_class"
    ]
    .value_counts()
    .to_dict()
)

hierarchy_group_counts = (
    prototype_roads_gdf[
        "hierarchy_group"
    ]
    .value_counts()
    .to_dict()
)

network_role_counts = (
    prototype_roads_gdf[
        "NetworkRole"
    ]
    .value_counts()
    .to_dict()
)

redundancy_counts = (
    prototype_roads_gdf[
        "RedundancyLabel"
    ]
    .value_counts()
    .to_dict()
)

vulnerability_counts = (
    prototype_roads_gdf[
        "VulnerabilityLabel"
    ]
    .value_counts()
    .to_dict()
)

scene_profile = {
    "scene_id": PROTOTYPE_SCENE_ID,

    # --------------------------------------------------------
    # Network scale
    # --------------------------------------------------------
    "road_record_count": total_road_records,
    "graph_node_count": total_nodes,
    "graph_edge_count": total_edges,
    "connected_component_count": (
        nx.number_connected_components(simple_graph)
    ),
    "largest_component_share": round(
        largest_component_share,
        4,
    ),

    # --------------------------------------------------------
    # Road hierarchy
    # --------------------------------------------------------
    "major_road_count": int(
        prototype_roads_gdf[
            "is_major_road"
        ].sum()
    ),
    "major_road_share": round(
        safe_share(
            prototype_roads_gdf[
                "is_major_road"
            ].sum(),
            total_road_records,
        ),
        4,
    ),
    "high_capacity_road_count": int(
        prototype_roads_gdf[
            "is_high_capacity_road"
        ].sum()
    ),

    # --------------------------------------------------------
    # Physical infrastructure
    # --------------------------------------------------------
    "physical_bridge_count": int(
        prototype_roads_gdf[
            "is_bridge_knowledge"
        ].sum()
    ),
    "tunnel_count": int(
        prototype_roads_gdf[
            "is_tunnel_knowledge"
        ].sum()
    ),
    "oneway_road_count": int(
        prototype_roads_gdf[
            "is_oneway_knowledge"
        ].sum()
    ),

    # --------------------------------------------------------
    # Topological structure
    # --------------------------------------------------------
    "articulation_point_count": int(
        len(network_articulation_points)
    ),
    "graph_bridge_record_count": int(
        prototype_roads_gdf[
            "is_network_bridge_edge"
        ].sum()
    ),
    "independent_cycle_count": int(
        len(cycle_basis)
    ),
    "cycle_road_record_count": int(
        prototype_roads_gdf[
            "participates_in_cycle"
        ].sum()
    ),

    # --------------------------------------------------------
    # Redundancy
    # --------------------------------------------------------
    "single_access_road_count": int(
        prototype_roads_gdf[
            "single_access_link"
        ].sum()
    ),
    "single_access_road_share": round(
        safe_share(
            prototype_roads_gdf[
                "single_access_link"
            ].sum(),
            total_road_records,
        ),
        4,
    ),
    "roads_with_alternate_routes_count": int(
        (
            prototype_roads_gdf[
                "local_edge_connectivity"
            ] >= 2
        ).sum()
    ),
    "roads_with_alternate_routes_share": round(
        safe_share(
            (
                prototype_roads_gdf[
                    "local_edge_connectivity"
                ] >= 2
            ).sum(),
            total_road_records,
        ),
        4,
    ),
    "mean_local_edge_connectivity": round(
        prototype_roads_gdf[
            "local_edge_connectivity"
        ].mean(),
        4,
    ),

    # --------------------------------------------------------
    # Criticality and vulnerability
    # --------------------------------------------------------
    "critical_transport_edge_count": int(
        prototype_roads_gdf[
            "critical_transport_edge"
        ].sum()
    ),
    "critical_transport_edge_share": round(
        safe_share(
            prototype_roads_gdf[
                "critical_transport_edge"
            ].sum(),
            total_road_records,
        ),
        4,
    ),
    "critical_low_redundancy_count": int(
        prototype_roads_gdf[
            "critical_low_redundancy_edge"
        ].sum()
    ),
    "bridge_bottleneck_count": int(
        prototype_roads_gdf[
            "bridge_bottleneck_edge"
        ].sum()
    ),
    "major_road_bottleneck_count": int(
        prototype_roads_gdf[
            "major_road_bottleneck_edge"
        ].sum()
    ),
    "high_topological_vulnerability_count": int(
        prototype_roads_gdf[
            "high_topological_vulnerability"
        ].sum()
    ),
    "mean_edge_criticality_score": round(
        prototype_roads_gdf[
            "edge_criticality_score"
        ].mean(),
        4,
    ),
    "mean_topological_vulnerability_score": round(
        prototype_roads_gdf[
            "topological_vulnerability_score"
        ].mean(),
        4,
    ),

    # --------------------------------------------------------
    # Category distributions
    # --------------------------------------------------------
    "road_class_distribution": road_class_counts,
    "hierarchy_group_distribution": hierarchy_group_counts,
    "network_role_distribution": network_role_counts,
    "redundancy_distribution": redundancy_counts,
    "vulnerability_distribution": vulnerability_counts,
}

scene_profile_df = pd.DataFrame(
    [scene_profile]
)

print("\nSCENE PROFILE SUMMARY")
print("-" * 72)

summary_fields = [
    "scene_id",
    "road_record_count",
    "graph_node_count",
    "graph_edge_count",
    "connected_component_count",
    "largest_component_share",
    "major_road_count",
    "physical_bridge_count",
    "articulation_point_count",
    "graph_bridge_record_count",
    "independent_cycle_count",
    "single_access_road_count",
    "roads_with_alternate_routes_count",
    "critical_transport_edge_count",
    "critical_low_redundancy_count",
    "bridge_bottleneck_count",
    "major_road_bottleneck_count",
    "high_topological_vulnerability_count",
]

print(
    scene_profile_df[
        summary_fields
    ]
    .T
    .to_string(
        header=False
    )
)

print("\nNETWORK ROLE DISTRIBUTION")
print("-" * 72)

print(
    prototype_roads_gdf[
        "NetworkRole"
    ]
    .value_counts()
    .to_string()
)

print("\nREDUNDANCY DISTRIBUTION")
print("-" * 72)

print(
    prototype_roads_gdf[
        "RedundancyLabel"
    ]
    .value_counts()
    .to_string()
)

print("\nVULNERABILITY DISTRIBUTION")
print("-" * 72)

print(
    prototype_roads_gdf[
        "VulnerabilityLabel"
    ]
    .value_counts()
    .to_string()
)

print("\n" + "=" * 72)
print("SCENE-LEVEL TRANSPORTATION PROFILE COMPLETE")
print("=" * 72)

BUILDING SCENE-LEVEL TRANSPORTATION PROFILE

SCENE PROFILE SUMMARY
------------------------------------------------------------------------
scene_id                              Spain_8565131
road_record_count                               902
graph_node_count                                449
graph_edge_count                                542
connected_component_count                         9
largest_component_share                      0.9354
major_road_count                                 23
physical_bridge_count                            33
articulation_point_count                        149
graph_bridge_record_count                       393
independent_cycle_count                         102
single_access_road_count                        393
roads_with_alternate_routes_count               509
critical_transport_edge_count                   180
critical_low_redundancy_count                    54
bridge_bottleneck_count                          21
major_road_bottleneck_count 

In [16]:
# ============================================================
# CELL 12: GENERATE SCENE-LEVEL TRANSPORTATION TOKEN
# ============================================================

print("=" * 72)
print("GENERATING SCENE-LEVEL TRANSPORTATION TOKEN")
print("=" * 72)


def classify_network_connectivity(
    largest_component_share,
):
    if largest_component_share >= 0.90:
        return "High"

    if largest_component_share >= 0.75:
        return "Moderate"

    return "Low"


def classify_exposure_share(
    share,
):
    if share >= 0.40:
        return "High"

    if share >= 0.20:
        return "Moderate"

    return "Low"


def classify_alternate_route_availability(
    share,
):
    if share >= 0.70:
        return "High"

    if share >= 0.40:
        return "Moderate"

    return "Low"


def classify_vulnerability_burden(
    share,
):
    if share >= 0.30:
        return "High"

    if share >= 0.15:
        return "Moderate"

    return "Low"


profile = scene_profile_df.iloc[0]

single_access_share = (
    profile["single_access_road_count"]
    / profile["road_record_count"]
    if profile["road_record_count"] > 0
    else 0.0
)

alternate_route_share = (
    profile["roads_with_alternate_routes_count"]
    / profile["road_record_count"]
    if profile["road_record_count"] > 0
    else 0.0
)

critical_edge_share = (
    profile["critical_transport_edge_count"]
    / profile["road_record_count"]
    if profile["road_record_count"] > 0
    else 0.0
)

high_vulnerability_share = (
    profile[
        "high_topological_vulnerability_count"
    ]
    / profile["road_record_count"]
    if profile["road_record_count"] > 0
    else 0.0
)

scene_semantics = {
    "NetworkConnectivity":
        classify_network_connectivity(
            profile["largest_component_share"]
        ),

    "SingleAccessExposure":
        classify_exposure_share(
            single_access_share
        ),

    "AlternateRouteAvailability":
        classify_alternate_route_availability(
            alternate_route_share
        ),

    "CriticalCorridorBurden":
        classify_vulnerability_burden(
            critical_edge_share
        ),

    "TopologicalVulnerabilityBurden":
        classify_vulnerability_burden(
            high_vulnerability_share
        ),

    "BridgeBottlenecks":
        (
            "Present"
            if profile["bridge_bottleneck_count"] > 0
            else "Absent"
        ),

    "MajorRoadBottlenecks":
        (
            "Present"
            if profile["major_road_bottleneck_count"] > 0
            else "Absent"
        ),

    "PhysicalBridges":
        (
            "Present"
            if profile["physical_bridge_count"] > 0
            else "Absent"
        ),

    "NetworkFragmentation":
        (
            "Present"
            if profile["connected_component_count"] > 1
            else "Absent"
        ),
}


def build_scene_transportation_token(
    semantic_dictionary,
):
    return " ".join(
        [
            f"<{key}:{value}>"
            for key, value
            in semantic_dictionary.items()
        ]
    )


scene_transportation_token = (
    build_scene_transportation_token(
        scene_semantics
    )
)

scene_profile_df[
    "scene_transportation_token"
] = scene_transportation_token

scene_profile_df[
    "single_access_share"
] = round(
    single_access_share,
    4,
)

scene_profile_df[
    "alternate_route_share"
] = round(
    alternate_route_share,
    4,
)

scene_profile_df[
    "critical_edge_share"
] = round(
    critical_edge_share,
    4,
)

scene_profile_df[
    "high_vulnerability_share"
] = round(
    high_vulnerability_share,
    4,
)

print("\nSCENE SEMANTIC CLASSIFICATION")
print("-" * 72)

for key, value in scene_semantics.items():
    print(
        f"{key:<35}: {value}"
    )

print("\nSCENE TRANSPORTATION TOKEN")
print("-" * 72)

print(
    scene_transportation_token
)

print("\nSCENE SHARE SUMMARY")
print("-" * 72)

print(
    f"Single-access share                 : "
    f"{single_access_share:.3f}"
)

print(
    f"Alternate-route share               : "
    f"{alternate_route_share:.3f}"
)

print(
    f"Critical transportation-edge share  : "
    f"{critical_edge_share:.3f}"
)

print(
    f"High-vulnerability edge share        : "
    f"{high_vulnerability_share:.3f}"
)

print("\n" + "=" * 72)
print("SCENE-LEVEL TRANSPORTATION TOKEN COMPLETE")
print("=" * 72)

GENERATING SCENE-LEVEL TRANSPORTATION TOKEN

SCENE SEMANTIC CLASSIFICATION
------------------------------------------------------------------------
NetworkConnectivity                : High
SingleAccessExposure               : High
AlternateRouteAvailability         : Moderate
CriticalCorridorBurden             : Moderate
TopologicalVulnerabilityBurden     : Moderate
BridgeBottlenecks                  : Present
MajorRoadBottlenecks               : Present
PhysicalBridges                    : Present
NetworkFragmentation               : Present

SCENE TRANSPORTATION TOKEN
------------------------------------------------------------------------
<NetworkConnectivity:High> <SingleAccessExposure:High> <AlternateRouteAvailability:Moderate> <CriticalCorridorBurden:Moderate> <TopologicalVulnerabilityBurden:Moderate> <BridgeBottlenecks:Present> <MajorRoadBottlenecks:Present> <PhysicalBridges:Present> <NetworkFragmentation:Present>

SCENE SHARE SUMMARY
-------------------------------------------

In [17]:
# ============================================================
# CELL 13: SAVE PROTOTYPE TRANSPORTATION KNOWLEDGE OUTPUTS
# ============================================================

from pathlib import Path
import json

print("=" * 72)
print("SAVING PROTOTYPE TRANSPORTATION KNOWLEDGE OUTPUTS")
print("=" * 72)

# ------------------------------------------------------------
# 1. Output directories
# ------------------------------------------------------------

from pathlib import Path

OUTPUT_DIR = Path.cwd() / "transportation_knowledge"

ROAD_OUTPUT_DIR = OUTPUT_DIR / "road_knowledge"
PROFILE_OUTPUT_DIR = OUTPUT_DIR / "scene_profiles"
TOKEN_OUTPUT_DIR = OUTPUT_DIR / "tokens"

for directory in [
    ROAD_OUTPUT_DIR,
    PROFILE_OUTPUT_DIR,
    TOKEN_OUTPUT_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

# ------------------------------------------------------------
# 2. Define output paths
# ------------------------------------------------------------

road_csv_path = (
    ROAD_OUTPUT_DIR
    / f"{PROTOTYPE_SCENE_ID}_road_knowledge.csv"
)

road_gpkg_path = (
    ROAD_OUTPUT_DIR
    / f"{PROTOTYPE_SCENE_ID}_road_knowledge.gpkg"
)

scene_profile_csv_path = (
    PROFILE_OUTPUT_DIR
    / f"{PROTOTYPE_SCENE_ID}_transportation_profile.csv"
)

scene_profile_json_path = (
    PROFILE_OUTPUT_DIR
    / f"{PROTOTYPE_SCENE_ID}_transportation_profile.json"
)

scene_token_txt_path = (
    TOKEN_OUTPUT_DIR
    / f"{PROTOTYPE_SCENE_ID}_transportation_token.txt"
)

# ------------------------------------------------------------
# 3. Save road-level transportation knowledge
# ------------------------------------------------------------

road_output_columns = [
    "u",
    "v",
    "key",
    "knowledge_highway_class",
    "hierarchy_score",
    "hierarchy_group",
    "HierarchyLabel",
    "NetworkRole",
    "RedundancyLabel",
    "VulnerabilityLabel",
    "is_bridge_knowledge",
    "is_tunnel_knowledge",
    "is_oneway_knowledge",
    "is_major_road",
    "is_high_capacity_road",
    "is_network_bridge_edge",
    "participates_in_cycle",
    "local_edge_connectivity",
    "single_access_link",
    "critical_transport_edge",
    "critical_low_redundancy_edge",
    "bridge_bottleneck_edge",
    "major_road_bottleneck_edge",
    "edge_criticality_score",
    "topological_vulnerability_score",
    "high_topological_vulnerability",
    "TransportationToken",
    "geometry",
]

existing_road_output_columns = [
    column
    for column in road_output_columns
    if column in prototype_roads_gdf.columns
]

prototype_road_knowledge_gdf = (
    prototype_roads_gdf[
        existing_road_output_columns
    ].copy()
)

prototype_road_knowledge_gdf.drop(
    columns="geometry"
).to_csv(
    road_csv_path,
    index=False,
)

prototype_road_knowledge_gdf.to_file(
    road_gpkg_path,
    layer="road_knowledge",
    driver="GPKG",
)

# ------------------------------------------------------------
# 4. Save scene profile
# ------------------------------------------------------------

scene_profile_df.to_csv(
    scene_profile_csv_path,
    index=False,
)

scene_profile_record = (
    scene_profile_df
    .iloc[0]
    .to_dict()
)

# Convert NumPy values to standard Python values
for key, value in scene_profile_record.items():

    if hasattr(value, "item"):
        scene_profile_record[key] = value.item()

with open(
    scene_profile_json_path,
    "w",
    encoding="utf-8",
) as json_file:

    json.dump(
        scene_profile_record,
        json_file,
        indent=4,
        ensure_ascii=False,
        default=str,
    )

# ------------------------------------------------------------
# 5. Save scene transportation token
# ------------------------------------------------------------

with open(
    scene_token_txt_path,
    "w",
    encoding="utf-8",
) as token_file:

    token_file.write(
        scene_transportation_token
    )

# ------------------------------------------------------------
# 6. Confirm outputs
# ------------------------------------------------------------

print("\nSAVED OUTPUTS")
print("-" * 72)

saved_outputs = {
    "Road knowledge CSV":
        road_csv_path,

    "Road knowledge GeoPackage":
        road_gpkg_path,

    "Scene profile CSV":
        scene_profile_csv_path,

    "Scene profile JSON":
        scene_profile_json_path,

    "Scene transportation token":
        scene_token_txt_path,
}

for output_name, output_path in saved_outputs.items():

    exists_status = (
        "SAVED"
        if output_path.exists()
        else "FAILED"
    )

    print(
        f"{exists_status:<8} | "
        f"{output_name:<30} | "
        f"{output_path}"
    )

print("\nOUTPUT RECORD COUNTS")
print("-" * 72)

print(
    f"Road knowledge records : "
    f"{len(prototype_road_knowledge_gdf):,}"
)

print(
    f"Scene profile records  : "
    f"{len(scene_profile_df):,}"
)

print(
    f"Scene token length     : "
    f"{len(scene_transportation_token):,} characters"
)

print("\n" + "=" * 72)
print("PROTOTYPE TRANSPORTATION KNOWLEDGE OUTPUTS SAVED")
print("=" * 72)

SAVING PROTOTYPE TRANSPORTATION KNOWLEDGE OUTPUTS



SAVED OUTPUTS
------------------------------------------------------------------------
SAVED    | Road knowledge CSV             | /home/adjeiowusu1/myproject/ResilientVLM/notebooks/transportation_knowledge/road_knowledge/Spain_8565131_road_knowledge.csv
SAVED    | Road knowledge GeoPackage      | /home/adjeiowusu1/myproject/ResilientVLM/notebooks/transportation_knowledge/road_knowledge/Spain_8565131_road_knowledge.gpkg
SAVED    | Scene profile CSV              | /home/adjeiowusu1/myproject/ResilientVLM/notebooks/transportation_knowledge/scene_profiles/Spain_8565131_transportation_profile.csv
SAVED    | Scene profile JSON             | /home/adjeiowusu1/myproject/ResilientVLM/notebooks/transportation_knowledge/scene_profiles/Spain_8565131_transportation_profile.json
SAVED    | Scene transportation token     | /home/adjeiowusu1/myproject/ResilientVLM/notebooks/transportation_knowledge/tokens/Spain_8565131_transportation_token.txt

OUTPUT RECORD COUNTS
----------------------------------

In [18]:
# ============================================================
# CELL 14: DEFINE REUSABLE TRANSPORTATION KNOWLEDGE PIPELINE
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx

from networkx.algorithms.connectivity import (
    local_edge_connectivity,
)

print("=" * 72)
print("DEFINING TRANSPORTATION KNOWLEDGE EXTRACTION PIPELINE")
print("=" * 72)


def safe_numeric(value, default=0.0):
    try:
        if pd.isna(value):
            return default
        return float(value)
    except (TypeError, ValueError):
        return default


def safe_bool(value):
    if isinstance(value, bool):
        return value

    if value is None or pd.isna(value):
        return False

    value_text = str(value).strip().lower()

    return value_text in {
        "true",
        "1",
        "yes",
        "y",
        "t",
    }


def percentile_rank(series):
    return (
        pd.Series(series)
        .rank(
            method="average",
            pct=True,
        )
        .fillna(0.0)
    )


def classify_hierarchy(score):
    if score >= 6:
        return "Motorway"
    if score >= 5:
        return "Trunk"
    if score >= 4:
        return "Primary"
    if score >= 3:
        return "Secondary"
    if score >= 2:
        return "Tertiary"
    return "Local"


def classify_redundancy(connectivity):
    if connectivity <= 1:
        return "None"
    if connectivity == 2:
        return "Low"
    if connectivity == 3:
        return "Moderate"
    return "High"


def classify_vulnerability(score):
    if score >= 0.80:
        return "Very High"
    if score >= 0.60:
        return "High"
    if score >= 0.40:
        return "Moderate"
    if score >= 0.20:
        return "Low"
    return "Very Low"


def determine_network_role(row):
    if row["bridge_bottleneck_edge"]:
        return "Critical Bridge"

    if row["critical_low_redundancy_edge"]:
        return "Critical Corridor"

    if row["critical_transport_edge"]:
        return "Primary Connector"

    if row["is_network_bridge_edge"]:
        return "Single Access"

    return "Local Access"


def clean_token_value(value):
    return (
        str(value)
        .replace(" ", "")
        .replace("-", "")
    )


def build_road_token(row):
    tokens = [
        (
            "<RoadHierarchy:"
            f"{clean_token_value(row['HierarchyLabel'])}>"
        ),
        (
            "<NetworkRole:"
            f"{clean_token_value(row['NetworkRole'])}>"
        ),
        (
            "<RouteRedundancy:"
            f"{clean_token_value(row['RedundancyLabel'])}>"
        ),
        (
            "<TopologicalVulnerability:"
            f"{clean_token_value(row['VulnerabilityLabel'])}>"
        ),
    ]

    optional_tokens = {
        "is_bridge_knowledge":
            "<PhysicalBridge:Yes>",

        "is_tunnel_knowledge":
            "<Tunnel:Yes>",

        "is_network_bridge_edge":
            "<GraphBridge:Yes>",

        "participates_in_cycle":
            "<AlternateRouteStructure:Yes>",

        "critical_transport_edge":
            "<CriticalTransportEdge:Yes>",

        "critical_low_redundancy_edge":
            "<CriticalLowRedundancy:Yes>",

        "bridge_bottleneck_edge":
            "<BridgeBottleneck:Yes>",

        "is_major_road":
            "<MajorRoad:Yes>",
    }

    for column, token in optional_tokens.items():
        if safe_bool(row.get(column, False)):
            tokens.append(token)

    return " ".join(tokens)


print("Reusable helper functions defined successfully.")
print("=" * 72)

DEFINING TRANSPORTATION KNOWLEDGE EXTRACTION PIPELINE
Reusable helper functions defined successfully.


In [19]:
# ============================================================
# CELL 15: DEFINE MAIN SCENE-PROCESSING FUNCTION
# ============================================================

print("=" * 72)
print("DEFINING MAIN TRANSPORTATION SCENE PROCESSOR")
print("=" * 72)


def process_transportation_scene(
    scene_id,
    graphml_path,
    roads_path,
    road_output_dir,
    profile_output_dir,
    token_output_dir,
    save_outputs=True,
):
    """
    Extract transportation knowledge for one scene.

    Parameters
    ----------
    scene_id : str
        Unique scene identifier.

    graphml_path : str or Path
        Path to the scene GraphML file.

    roads_path : str or Path
        Path to the road GeoPackage or other vector layer.

    road_output_dir : str or Path
        Directory for road-level knowledge outputs.

    profile_output_dir : str or Path
        Directory for scene-level profiles.

    token_output_dir : str or Path
        Directory for scene-level tokens.

    save_outputs : bool
        Whether to save the generated outputs.

    Returns
    -------
    dict
        Processing result containing:
        - status
        - scene_id
        - road_knowledge
        - scene_profile
        - scene_token
        - processing_summary
    """

    graphml_path = Path(graphml_path)
    roads_path = Path(roads_path)

    road_output_dir = Path(road_output_dir)
    profile_output_dir = Path(profile_output_dir)
    token_output_dir = Path(token_output_dir)

    print("\n" + "=" * 72)
    print(f"PROCESSING SCENE: {scene_id}")
    print("=" * 72)

    # --------------------------------------------------------
    # 1. Validate inputs
    # --------------------------------------------------------

    if not graphml_path.exists():
        raise FileNotFoundError(
            f"GraphML file not found: {graphml_path}"
        )

    if not roads_path.exists():
        raise FileNotFoundError(
            f"Road layer not found: {roads_path}"
        )

    # --------------------------------------------------------
    # 2. Load graph and road records
    # --------------------------------------------------------

    graph = nx.read_graphml(
        graphml_path
    )

    roads_gdf = gpd.read_file(
        roads_path,
        layer="roads",
    )

    if roads_gdf.empty:
        raise ValueError(
            f"No road records were found for {scene_id}."
        )

    print(
        f"Loaded graph        : "
        f"{graph.number_of_nodes():,} nodes, "
        f"{graph.number_of_edges():,} edges"
    )

    print(
        f"Loaded road records : "
        f"{len(roads_gdf):,}"
    )

    # --------------------------------------------------------
    # 3. Identify endpoint columns
    # --------------------------------------------------------

    required_endpoint_columns = {
        "u",
        "v",
    }

    missing_endpoint_columns = (
        required_endpoint_columns
        - set(roads_gdf.columns)
    )

    if missing_endpoint_columns:
        raise ValueError(
            "Road layer is missing endpoint columns: "
            f"{sorted(missing_endpoint_columns)}"
        )

    if "key" not in roads_gdf.columns:
        roads_gdf["key"] = 0

    roads_gdf["u_string"] = (
        roads_gdf["u"].astype(str)
    )

    roads_gdf["v_string"] = (
        roads_gdf["v"].astype(str)
    )

    # --------------------------------------------------------
    # 4. Build a clean undirected simple graph
    # --------------------------------------------------------

    simple_graph = nx.Graph()

    simple_graph.add_nodes_from(
        graph.nodes(data=True)
    )

    if graph.is_multigraph():

        for u, v, key, edge_data in graph.edges(
            keys=True,
            data=True,
        ):
            if str(u) != str(v):
                simple_graph.add_edge(
                    u,
                    v,
                    **edge_data,
                )

    else:

        for u, v, edge_data in graph.edges(
            data=True
        ):
            if str(u) != str(v):
                simple_graph.add_edge(
                    u,
                    v,
                    **edge_data,
                )

    simple_graph.remove_edges_from(
        nx.selfloop_edges(simple_graph)
    )

    graph_node_lookup = {
        str(node): node
        for node in simple_graph.nodes
    }

    # --------------------------------------------------------
    # 5. Normalize highway classes
    # --------------------------------------------------------

    highway_source_column = None

    highway_candidates = [
        "knowledge_highway_class",
        "highway",
        "highway_class",
        "road_class",
    ]

    for candidate in highway_candidates:
        if candidate in roads_gdf.columns:
            highway_source_column = candidate
            break

    if highway_source_column is None:
        roads_gdf["knowledge_highway_class"] = (
            "unclassified"
        )

    else:
        roads_gdf["knowledge_highway_class"] = (
            roads_gdf[
                highway_source_column
            ].apply(
                normalize_highway_class
            )
        )

    roads_gdf["hierarchy_score"] = (
        roads_gdf[
            "knowledge_highway_class"
        ].apply(
            road_hierarchy_score
        )
    )

    roads_gdf["hierarchy_group"] = (
        roads_gdf[
            "knowledge_highway_class"
        ].apply(
            road_hierarchy_group
        )
    )

    # --------------------------------------------------------
    # 6. Infrastructure flags
    # --------------------------------------------------------

    for source_column, output_column in [
        ("bridge", "is_bridge_knowledge"),
        ("tunnel", "is_tunnel_knowledge"),
        ("oneway", "is_oneway_knowledge"),
    ]:

        if source_column in roads_gdf.columns:
            roads_gdf[output_column] = (
                roads_gdf[
                    source_column
                ].apply(
                    safe_bool
                )
            )
        else:
            roads_gdf[output_column] = False

    roads_gdf["is_major_road"] = (
        roads_gdf["hierarchy_score"] >= 4
    )

    roads_gdf["is_high_capacity_road"] = (
        roads_gdf["hierarchy_score"] >= 5
    )

    roads_gdf["is_critical_link"] = (
        roads_gdf[
            [
                "is_bridge_knowledge",
                "is_tunnel_knowledge",
                "is_major_road",
            ]
        ]
        .any(axis=1)
    )

    # --------------------------------------------------------
    # 7. Node centrality
    # --------------------------------------------------------

    degree_dictionary = dict(
        simple_graph.degree()
    )

    degree_centrality_dictionary = (
        nx.degree_centrality(
            simple_graph
        )
        if simple_graph.number_of_nodes() > 1
        else {
            node: 0.0
            for node in simple_graph.nodes
        }
    )

    if simple_graph.number_of_edges() > 0:

        betweenness_dictionary = (
            nx.betweenness_centrality(
                simple_graph,
                normalized=True,
            )
        )

        closeness_dictionary = (
            nx.closeness_centrality(
                simple_graph
            )
        )

    else:

        betweenness_dictionary = {
            node: 0.0
            for node in simple_graph.nodes
        }

        closeness_dictionary = {
            node: 0.0
            for node in simple_graph.nodes
        }

    articulation_points = set(
        nx.articulation_points(
            simple_graph
        )
    )

    node_knowledge_df = pd.DataFrame(
        {
            "node": list(
                simple_graph.nodes
            )
        }
    )

    node_knowledge_df["degree"] = (
        node_knowledge_df["node"]
        .map(degree_dictionary)
        .fillna(0)
    )

    node_knowledge_df[
        "degree_centrality"
    ] = (
        node_knowledge_df["node"]
        .map(
            degree_centrality_dictionary
        )
        .fillna(0.0)
    )

    node_knowledge_df[
        "betweenness_centrality"
    ] = (
        node_knowledge_df["node"]
        .map(
            betweenness_dictionary
        )
        .fillna(0.0)
    )

    node_knowledge_df[
        "closeness_centrality"
    ] = (
        node_knowledge_df["node"]
        .map(
            closeness_dictionary
        )
        .fillna(0.0)
    )

    node_knowledge_df[
        "degree_percentile"
    ] = percentile_rank(
        node_knowledge_df[
            "degree_centrality"
        ]
    )

    node_knowledge_df[
        "betweenness_percentile"
    ] = percentile_rank(
        node_knowledge_df[
            "betweenness_centrality"
        ]
    )

    node_knowledge_df[
        "closeness_percentile"
    ] = percentile_rank(
        node_knowledge_df[
            "closeness_centrality"
        ]
    )

    node_knowledge_df[
        "node_importance_score"
    ] = (
        0.50
        * node_knowledge_df[
            "betweenness_percentile"
        ]
        + 0.25
        * node_knowledge_df[
            "closeness_percentile"
        ]
        + 0.25
        * node_knowledge_df[
            "degree_percentile"
        ]
    )

    node_knowledge_df[
        "is_articulation_point"
    ] = (
        node_knowledge_df["node"]
        .isin(articulation_points)
    )

    node_knowledge_df[
        "is_critical_node"
    ] = (
        (
            node_knowledge_df[
                "node_importance_score"
            ]
            >= 0.80
        )
        |
        node_knowledge_df[
            "is_articulation_point"
        ]
    )

    node_importance_lookup = dict(
        zip(
            node_knowledge_df[
                "node"
            ].astype(str),
            node_knowledge_df[
                "node_importance_score"
            ],
        )
    )

    node_betweenness_lookup = dict(
        zip(
            node_knowledge_df[
                "node"
            ].astype(str),
            node_knowledge_df[
                "betweenness_centrality"
            ],
        )
    )

    critical_node_strings = set(
        node_knowledge_df.loc[
            node_knowledge_df[
                "is_critical_node"
            ],
            "node",
        ].astype(str)
    )

    articulation_point_strings = {
        str(node)
        for node in articulation_points
    }

    # --------------------------------------------------------
    # 8. Transfer node importance to road records
    # --------------------------------------------------------

    roads_gdf[
        "u_node_importance"
    ] = (
        roads_gdf["u_string"]
        .map(node_importance_lookup)
        .fillna(0.0)
    )

    roads_gdf[
        "v_node_importance"
    ] = (
        roads_gdf["v_string"]
        .map(node_importance_lookup)
        .fillna(0.0)
    )

    roads_gdf[
        "mean_endpoint_importance"
    ] = (
        roads_gdf[
            [
                "u_node_importance",
                "v_node_importance",
            ]
        ]
        .mean(axis=1)
    )

    roads_gdf[
        "max_endpoint_importance"
    ] = (
        roads_gdf[
            [
                "u_node_importance",
                "v_node_importance",
            ]
        ]
        .max(axis=1)
    )

    roads_gdf[
        "u_node_betweenness"
    ] = (
        roads_gdf["u_string"]
        .map(node_betweenness_lookup)
        .fillna(0.0)
    )

    roads_gdf[
        "v_node_betweenness"
    ] = (
        roads_gdf["v_string"]
        .map(node_betweenness_lookup)
        .fillna(0.0)
    )

    roads_gdf[
        "mean_endpoint_betweenness"
    ] = (
        roads_gdf[
            [
                "u_node_betweenness",
                "v_node_betweenness",
            ]
        ]
        .mean(axis=1)
    )

    roads_gdf[
        "touches_critical_node"
    ] = (
        roads_gdf["u_string"].isin(
            critical_node_strings
        )
        |
        roads_gdf["v_string"].isin(
            critical_node_strings
        )
    )

    roads_gdf[
        "touches_articulation_point"
    ] = (
        roads_gdf["u_string"].isin(
            articulation_point_strings
        )
        |
        roads_gdf["v_string"].isin(
            articulation_point_strings
        )
    )

    # --------------------------------------------------------
    # 9. Initial edge criticality
    # --------------------------------------------------------

    hierarchy_normalized = (
        roads_gdf[
            "hierarchy_score"
        ]
        .clip(lower=1, upper=6)
        / 6.0
    )

    roads_gdf[
        "edge_criticality_score"
    ] = (
        0.40 * hierarchy_normalized
        + 0.35
        * roads_gdf[
            "max_endpoint_importance"
        ]
        + 0.15
        * roads_gdf[
            "is_bridge_knowledge"
        ].astype(float)
        + 0.05
        * roads_gdf[
            "is_tunnel_knowledge"
        ].astype(float)
        + 0.05
        * roads_gdf[
            "touches_articulation_point"
        ].astype(float)
    ).clip(
        lower=0.0,
        upper=1.0,
    )

    criticality_threshold = (
        roads_gdf[
            "edge_criticality_score"
        ]
        .quantile(0.80)
    )

    roads_gdf[
        "critical_transport_edge"
    ] = (
        roads_gdf[
            "edge_criticality_score"
        ]
        >= criticality_threshold
    )

    # --------------------------------------------------------
    # 10. Graph bridges and cycles
    # --------------------------------------------------------

    network_bridge_pairs = {
        frozenset(
            (
                str(u),
                str(v),
            )
        )
        for u, v in nx.bridges(
            simple_graph
        )
    }

    roads_gdf[
        "canonical_endpoint_pair"
    ] = [
        frozenset(
            (
                str(u),
                str(v),
            )
        )
        for u, v in zip(
            roads_gdf["u"],
            roads_gdf["v"],
        )
    ]

    roads_gdf[
        "is_network_bridge_edge"
    ] = (
        roads_gdf[
            "canonical_endpoint_pair"
        ]
        .isin(network_bridge_pairs)
    )

    cycle_basis = nx.cycle_basis(
        simple_graph
    )

    cycle_edge_pairs = set()

    for cycle in cycle_basis:

        if len(cycle) < 2:
            continue

        for index in range(
            len(cycle)
        ):

            u_node = cycle[index]

            v_node = cycle[
                (index + 1)
                % len(cycle)
            ]

            cycle_edge_pairs.add(
                frozenset(
                    (
                        str(u_node),
                        str(v_node),
                    )
                )
            )

    roads_gdf[
        "participates_in_cycle"
    ] = (
        roads_gdf[
            "canonical_endpoint_pair"
        ]
        .isin(cycle_edge_pairs)
    )

    # --------------------------------------------------------
    # 11. Local edge connectivity
    # --------------------------------------------------------

    unique_pairs_df = (
        roads_gdf[
            [
                "u_string",
                "v_string",
                "canonical_endpoint_pair",
            ]
        ]
        .drop_duplicates(
            subset=[
                "canonical_endpoint_pair"
            ]
        )
        .reset_index(drop=True)
    )

    connectivity_lookup = {}

    total_pairs = len(
        unique_pairs_df
    )

    print(
        f"Computing local connectivity for "
        f"{total_pairs:,} unique undirected pairs..."
    )

    for pair_index, pair_row in (
        unique_pairs_df.iterrows()
    ):

        u_string = pair_row[
            "u_string"
        ]

        v_string = pair_row[
            "v_string"
        ]

        u_node = graph_node_lookup.get(
            u_string
        )

        v_node = graph_node_lookup.get(
            v_string
        )

        canonical_pair = pair_row[
            "canonical_endpoint_pair"
        ]

        if (
            u_node is None
            or v_node is None
            or u_node == v_node
        ):
            connectivity_value = 0

        else:
            try:
                connectivity_value = (
                    local_edge_connectivity(
                        simple_graph,
                        u_node,
                        v_node,
                        cutoff=4,
                    )
                )

            except (
                nx.NetworkXError,
                nx.NodeNotFound,
                nx.NetworkXNoPath,
                ValueError,
            ):
                connectivity_value = 0

        connectivity_lookup[
            canonical_pair
        ] = int(
            connectivity_value
        )

        if (
            (pair_index + 1) % 100 == 0
            or pair_index + 1
            == total_pairs
        ):
            print(
                f"Processed "
                f"{pair_index + 1:,}/"
                f"{total_pairs:,}"
            )

    roads_gdf[
        "local_edge_connectivity"
    ] = (
        roads_gdf[
            "canonical_endpoint_pair"
        ]
        .map(
            connectivity_lookup
        )
        .fillna(0)
        .astype(int)
    )

    roads_gdf[
        "single_access_link"
    ] = (
        roads_gdf[
            "local_edge_connectivity"
        ]
        <= 1
    )

    # --------------------------------------------------------
    # 12. Bottleneck indicators
    # --------------------------------------------------------

    roads_gdf[
        "critical_low_redundancy_edge"
    ] = (
        roads_gdf[
            "critical_transport_edge"
        ]
        &
        roads_gdf[
            "single_access_link"
        ]
    )

    roads_gdf[
        "bridge_bottleneck_edge"
    ] = (
        roads_gdf[
            "is_bridge_knowledge"
        ]
        &
        roads_gdf[
            "is_network_bridge_edge"
        ]
    )

    roads_gdf[
        "major_road_bottleneck_edge"
    ] = (
        roads_gdf[
            "is_major_road"
        ]
        &
        roads_gdf[
            "is_network_bridge_edge"
        ]
    )

    redundancy_penalty = (
        1.0
        - (
            roads_gdf[
                "local_edge_connectivity"
            ]
            .clip(lower=1, upper=4)
            - 1
        )
        / 3.0
    )

    roads_gdf[
        "topological_vulnerability_score"
    ] = (
        0.45
        * roads_gdf[
            "edge_criticality_score"
        ]
        + 0.25
        * roads_gdf[
            "is_network_bridge_edge"
        ].astype(float)
        + 0.20
        * redundancy_penalty
        + 0.10
        * (
            ~roads_gdf[
                "participates_in_cycle"
            ]
        ).astype(float)
    ).clip(
        lower=0.0,
        upper=1.0,
    )

    vulnerability_threshold = (
        roads_gdf[
            "topological_vulnerability_score"
        ]
        .quantile(0.80)
    )

    roads_gdf[
        "high_topological_vulnerability"
    ] = (
        roads_gdf[
            "topological_vulnerability_score"
        ]
        >= vulnerability_threshold
    )

    # --------------------------------------------------------
    # 13. Semantic labels
    # --------------------------------------------------------

    roads_gdf[
        "HierarchyLabel"
    ] = (
        roads_gdf[
            "hierarchy_score"
        ]
        .apply(
            classify_hierarchy
        )
    )

    roads_gdf[
        "RedundancyLabel"
    ] = (
        roads_gdf[
            "local_edge_connectivity"
        ]
        .apply(
            classify_redundancy
        )
    )

    roads_gdf[
        "VulnerabilityLabel"
    ] = (
        roads_gdf[
            "topological_vulnerability_score"
        ]
        .apply(
            classify_vulnerability
        )
    )

    roads_gdf[
        "NetworkRole"
    ] = roads_gdf.apply(
        determine_network_role,
        axis=1,
    )

    roads_gdf[
        "TransportationToken"
    ] = roads_gdf.apply(
        build_road_token,
        axis=1,
    )

    # --------------------------------------------------------
    # 14. Scene profile
    # --------------------------------------------------------

    total_road_records = len(
        roads_gdf
    )

    total_nodes = (
        simple_graph.number_of_nodes()
    )

    total_edges = (
        simple_graph.number_of_edges()
    )

    connected_components = list(
        nx.connected_components(
            simple_graph
        )
    )

    connected_component_count = len(
        connected_components
    )

    largest_component_nodes = (
        max(
            (
                len(component)
                for component
                in connected_components
            ),
            default=0,
        )
    )

    largest_component_share = (
        largest_component_nodes
        / total_nodes
        if total_nodes > 0
        else 0.0
    )

    def count_true(column):
        return int(
            roads_gdf[column]
            .fillna(False)
            .astype(bool)
            .sum()
        )

    def calculate_share(count):
        if total_road_records == 0:
            return 0.0

        return float(
            count
        ) / float(
            total_road_records
        )

    major_road_count = count_true(
        "is_major_road"
    )

    physical_bridge_count = count_true(
        "is_bridge_knowledge"
    )

    single_access_count = count_true(
        "single_access_link"
    )

    alternate_route_count = int(
        (
            roads_gdf[
                "local_edge_connectivity"
            ]
            >= 2
        ).sum()
    )

    critical_edge_count = count_true(
        "critical_transport_edge"
    )

    high_vulnerability_count = count_true(
        "high_topological_vulnerability"
    )

    scene_profile = {
        "scene_id":
            scene_id,

        "road_record_count":
            total_road_records,

        "graph_node_count":
            total_nodes,

        "graph_edge_count":
            total_edges,

        "connected_component_count":
            connected_component_count,

        "largest_component_nodes":
            largest_component_nodes,

        "largest_component_share":
            round(
                largest_component_share,
                4,
            ),

        "major_road_count":
            major_road_count,

        "major_road_share":
            round(
                calculate_share(
                    major_road_count
                ),
                4,
            ),

        "high_capacity_road_count":
            count_true(
                "is_high_capacity_road"
            ),

        "physical_bridge_count":
            physical_bridge_count,

        "tunnel_count":
            count_true(
                "is_tunnel_knowledge"
            ),

        "oneway_road_count":
            count_true(
                "is_oneway_knowledge"
            ),

        "articulation_point_count":
            len(
                articulation_points
            ),

        "critical_node_count":
            int(
                node_knowledge_df[
                    "is_critical_node"
                ].sum()
            ),

        "graph_bridge_record_count":
            count_true(
                "is_network_bridge_edge"
            ),

        "independent_cycle_count":
            len(
                cycle_basis
            ),

        "cycle_road_record_count":
            count_true(
                "participates_in_cycle"
            ),

        "single_access_road_count":
            single_access_count,

        "single_access_road_share":
            round(
                calculate_share(
                    single_access_count
                ),
                4,
            ),

        "roads_with_alternate_routes_count":
            alternate_route_count,

        "roads_with_alternate_routes_share":
            round(
                calculate_share(
                    alternate_route_count
                ),
                4,
            ),

        "mean_local_edge_connectivity":
            round(
                roads_gdf[
                    "local_edge_connectivity"
                ].mean(),
                4,
            ),

        "critical_transport_edge_count":
            critical_edge_count,

        "critical_transport_edge_share":
            round(
                calculate_share(
                    critical_edge_count
                ),
                4,
            ),

        "critical_low_redundancy_count":
            count_true(
                "critical_low_redundancy_edge"
            ),

        "bridge_bottleneck_count":
            count_true(
                "bridge_bottleneck_edge"
            ),

        "major_road_bottleneck_count":
            count_true(
                "major_road_bottleneck_edge"
            ),

        "high_topological_vulnerability_count":
            high_vulnerability_count,

        "high_topological_vulnerability_share":
            round(
                calculate_share(
                    high_vulnerability_count
                ),
                4,
            ),

        "mean_edge_criticality_score":
            round(
                roads_gdf[
                    "edge_criticality_score"
                ].mean(),
                4,
            ),

        "mean_topological_vulnerability_score":
            round(
                roads_gdf[
                    "topological_vulnerability_score"
                ].mean(),
                4,
            ),

        "road_class_distribution":
            roads_gdf[
                "knowledge_highway_class"
            ]
            .value_counts()
            .to_dict(),

        "hierarchy_group_distribution":
            roads_gdf[
                "hierarchy_group"
            ]
            .value_counts()
            .to_dict(),

        "network_role_distribution":
            roads_gdf[
                "NetworkRole"
            ]
            .value_counts()
            .to_dict(),

        "redundancy_distribution":
            roads_gdf[
                "RedundancyLabel"
            ]
            .value_counts()
            .to_dict(),

        "vulnerability_distribution":
            roads_gdf[
                "VulnerabilityLabel"
            ]
            .value_counts()
            .to_dict(),
    }

    # --------------------------------------------------------
    # 15. Scene semantic classification
    # --------------------------------------------------------

    single_access_share = (
        scene_profile[
            "single_access_road_share"
        ]
    )

    alternate_route_share = (
        scene_profile[
            "roads_with_alternate_routes_share"
        ]
    )

    critical_edge_share = (
        scene_profile[
            "critical_transport_edge_share"
        ]
    )

    high_vulnerability_share = (
        scene_profile[
            "high_topological_vulnerability_share"
        ]
    )

    network_connectivity_label = (
        "High"
        if largest_component_share >= 0.90
        else "Moderate"
        if largest_component_share >= 0.75
        else "Low"
    )

    single_access_label = (
        "High"
        if single_access_share >= 0.40
        else "Moderate"
        if single_access_share >= 0.20
        else "Low"
    )

    alternate_route_label = (
        "High"
        if alternate_route_share >= 0.70
        else "Moderate"
        if alternate_route_share >= 0.40
        else "Low"
    )

    critical_corridor_label = (
        "High"
        if critical_edge_share >= 0.30
        else "Moderate"
        if critical_edge_share >= 0.15
        else "Low"
    )

    vulnerability_burden_label = (
        "High"
        if high_vulnerability_share >= 0.30
        else "Moderate"
        if high_vulnerability_share >= 0.15
        else "Low"
    )

    scene_semantics = {
        "NetworkConnectivity":
            network_connectivity_label,

        "SingleAccessExposure":
            single_access_label,

        "AlternateRouteAvailability":
            alternate_route_label,

        "CriticalCorridorBurden":
            critical_corridor_label,

        "TopologicalVulnerabilityBurden":
            vulnerability_burden_label,

        "BridgeBottlenecks":
            (
                "Present"
                if scene_profile[
                    "bridge_bottleneck_count"
                ] > 0
                else "Absent"
            ),

        "MajorRoadBottlenecks":
            (
                "Present"
                if scene_profile[
                    "major_road_bottleneck_count"
                ] > 0
                else "Absent"
            ),

        "PhysicalBridges":
            (
                "Present"
                if physical_bridge_count > 0
                else "Absent"
            ),

        "NetworkFragmentation":
            (
                "Present"
                if connected_component_count > 1
                else "Absent"
            ),
    }

    scene_token = " ".join(
        [
            f"<{key}:{value}>"
            for key, value
            in scene_semantics.items()
        ]
    )

    scene_profile[
        "scene_transportation_token"
    ] = scene_token

    scene_profile_df = pd.DataFrame(
        [scene_profile]
    )

    # --------------------------------------------------------
    # 16. Save outputs
    # --------------------------------------------------------

    if save_outputs:

        for output_directory in [
            road_output_dir,
            profile_output_dir,
            token_output_dir,
        ]:
            output_directory.mkdir(
                parents=True,
                exist_ok=True,
            )

        road_csv_path = (
            road_output_dir
            / f"{scene_id}_road_knowledge.csv"
        )

        road_gpkg_path = (
            road_output_dir
            / f"{scene_id}_road_knowledge.gpkg"
        )

        node_csv_path = (
            road_output_dir
            / f"{scene_id}_node_knowledge.csv"
        )

        profile_csv_path = (
            profile_output_dir
            / f"{scene_id}_transportation_profile.csv"
        )

        profile_json_path = (
            profile_output_dir
            / f"{scene_id}_transportation_profile.json"
        )

        token_path = (
            token_output_dir
            / f"{scene_id}_transportation_token.txt"
        )

        # Frozensets cannot be saved directly.
        road_output_gdf = roads_gdf.drop(
            columns=[
                "canonical_endpoint_pair"
            ],
            errors="ignore",
        ).copy()

        road_output_gdf.drop(
            columns="geometry",
            errors="ignore",
        ).to_csv(
            road_csv_path,
            index=False,
        )

        road_output_gdf.to_file(
            road_gpkg_path,
            layer="road_knowledge",
            driver="GPKG",
        )

        node_knowledge_df.to_csv(
            node_csv_path,
            index=False,
        )

        scene_profile_df.to_csv(
            profile_csv_path,
            index=False,
        )

        with open(
            profile_json_path,
            "w",
            encoding="utf-8",
        ) as json_file:

            json.dump(
                scene_profile,
                json_file,
                indent=4,
                ensure_ascii=False,
                default=str,
            )

        with open(
            token_path,
            "w",
            encoding="utf-8",
        ) as token_file:

            token_file.write(
                scene_token
            )

    # --------------------------------------------------------
    # 17. Processing summary
    # --------------------------------------------------------

    processing_summary = {
        "scene_id":
            scene_id,

        "status":
            "SUCCESS",

        "road_records":
            total_road_records,

        "graph_nodes":
            total_nodes,

        "graph_edges":
            total_edges,

        "critical_edges":
            critical_edge_count,

        "single_access_edges":
            single_access_count,

        "bridge_bottlenecks":
            scene_profile[
                "bridge_bottleneck_count"
            ],

        "major_road_bottlenecks":
            scene_profile[
                "major_road_bottleneck_count"
            ],

        "scene_token_length":
            len(scene_token),
    }

    print("\nSCENE PROCESSING SUMMARY")
    print("-" * 72)

    for key, value in (
        processing_summary.items()
    ):
        print(
            f"{key:<30}: {value}"
        )

    print(
        f"\nSCENE COMPLETE: {scene_id}"
    )

    return {
        "status":
            "SUCCESS",

        "scene_id":
            scene_id,

        "road_knowledge":
            roads_gdf,

        "node_knowledge":
            node_knowledge_df,

        "scene_profile":
            scene_profile_df,

        "scene_token":
            scene_token,

        "processing_summary":
            processing_summary,
    }


print(
    "Main scene-processing function defined successfully."
)

print("=" * 72)

DEFINING MAIN TRANSPORTATION SCENE PROCESSOR
Main scene-processing function defined successfully.


In [20]:
from pathlib import Path

project_root = Path.cwd()

print("Current notebook folder:")
print(project_root)
print()

print("Searching for GraphML files...\n")

graphml_files = list(project_root.rglob("*.graphml"))

print(f"Found {len(graphml_files)} GraphML files\n")

for f in graphml_files:
    print(f)

Current notebook folder:
/home/adjeiowusu1/myproject/ResilientVLM/notebooks

Searching for GraphML files...

Found 0 GraphML files



In [21]:
from pathlib import Path

print("Searching for GeoPackage files...\n")

gpkg_files = list(Path.cwd().rglob("*.gpkg"))

print(f"Found {len(gpkg_files)} GeoPackage files\n")

for f in gpkg_files:
    print(f)

Searching for GeoPackage files...

Found 1 GeoPackage files

/home/adjeiowusu1/myproject/ResilientVLM/notebooks/transportation_knowledge/road_knowledge/Spain_8565131_road_knowledge.gpkg


In [22]:
# ============================================================
# LOCATE ALL GRAPHML AND GEOPACKAGE FILES IN THE PROJECT
# ============================================================

from pathlib import Path

# Current location:
# ResilientVLM/notebooks
NOTEBOOK_DIR = Path.cwd()

# Move one level up:
# ResilientVLM
PROJECT_ROOT = NOTEBOOK_DIR.parent

print("=" * 72)
print("PROJECT FILE SEARCH")
print("=" * 72)

print(f"Notebook directory : {NOTEBOOK_DIR}")
print(f"Project root       : {PROJECT_ROOT}")

graphml_files = sorted(
    PROJECT_ROOT.rglob("*.graphml")
)

gpkg_files = sorted(
    PROJECT_ROOT.rglob("*.gpkg")
)

print("\n" + "-" * 72)
print(f"GRAPHML FILES FOUND: {len(graphml_files)}")
print("-" * 72)

for file_path in graphml_files:
    print(file_path)

print("\n" + "-" * 72)
print(f"GEOPACKAGE FILES FOUND: {len(gpkg_files)}")
print("-" * 72)

for file_path in gpkg_files:
    print(file_path)

PROJECT FILE SEARCH
Notebook directory : /home/adjeiowusu1/myproject/ResilientVLM/notebooks
Project root       : /home/adjeiowusu1/myproject/ResilientVLM



------------------------------------------------------------------------
GRAPHML FILES FOUND: 26
------------------------------------------------------------------------
/home/adjeiowusu1/myproject/ResilientVLM/data/processed/transportation_knowledge_graphs/graphml/Ghana_141910_transportation_graph.graphml
/home/adjeiowusu1/myproject/ResilientVLM/data/processed/transportation_knowledge_graphs/graphml/India_1018327_transportation_graph.graphml
/home/adjeiowusu1/myproject/ResilientVLM/data/processed/transportation_knowledge_graphs/graphml/India_1050276_transportation_graph.graphml
/home/adjeiowusu1/myproject/ResilientVLM/data/processed/transportation_knowledge_graphs/graphml/India_1068117_transportation_graph.graphml
/home/adjeiowusu1/myproject/ResilientVLM/data/processed/transportation_knowledge_graphs/graphml/India_285297_transportation_graph.graphml
/home/adjeiowusu1/myproject/ResilientVLM/data/processed/transportation_knowledge_graphs/graphml/India_383430_transportation_graph.graphm

In [23]:
# ============================================================
# CELL 16: BUILD TRANSPORTATION SCENE INVENTORY
# ============================================================

from pathlib import Path
import pandas as pd

print("=" * 72)
print("BUILDING TRANSPORTATION SCENE INVENTORY")
print("=" * 72)

PROJECT_ROOT = Path.cwd().parent

GRAPHML_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "transportation_knowledge_graphs"
    / "graphml"
)

GEOPACKAGE_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "transportation_knowledge_graphs"
    / "geopackages"
)

print(f"GraphML directory    : {GRAPHML_DIR}")
print(f"GeoPackage directory : {GEOPACKAGE_DIR}")

graphml_files = sorted(
    GRAPHML_DIR.glob("*_transportation_graph.graphml")
)

scene_records = []

for graphml_path in graphml_files:

    scene_id = graphml_path.stem.replace(
        "_transportation_graph",
        "",
    )

    gpkg_path = (
        GEOPACKAGE_DIR
        / f"{scene_id}_transportation_knowledge.gpkg"
    )

    scene_records.append(
        {
            "scene_id": scene_id,
            "graphml_path": graphml_path,
            "roads_path": gpkg_path,
            "graphml_exists": graphml_path.exists(),
            "roads_exists": gpkg_path.exists(),
        }
    )

scene_inventory_df = pd.DataFrame(
    scene_records
)

scene_inventory_df["ready_to_process"] = (
    scene_inventory_df["graphml_exists"]
    & scene_inventory_df["roads_exists"]
)

print("\nSCENE INVENTORY")
print("-" * 72)

display(
    scene_inventory_df[
        [
            "scene_id",
            "graphml_exists",
            "roads_exists",
            "ready_to_process",
        ]
    ]
)

print("\nINVENTORY SUMMARY")
print("-" * 72)

print(
    f"GraphML files found     : "
    f"{scene_inventory_df['graphml_exists'].sum()}"
)

print(
    f"GeoPackages matched     : "
    f"{scene_inventory_df['roads_exists'].sum()}"
)

print(
    f"Scenes ready to process : "
    f"{scene_inventory_df['ready_to_process'].sum()}"
)

missing_scenes_df = scene_inventory_df.loc[
    ~scene_inventory_df["ready_to_process"]
].copy()

if missing_scenes_df.empty:
    print("\nAll transportation scenes are ready.")
else:
    print("\nMissing or unmatched files:")
    display(missing_scenes_df)

print("=" * 72)

BUILDING TRANSPORTATION SCENE INVENTORY
GraphML directory    : /home/adjeiowusu1/myproject/ResilientVLM/data/processed/transportation_knowledge_graphs/graphml
GeoPackage directory : /home/adjeiowusu1/myproject/ResilientVLM/data/processed/transportation_knowledge_graphs/geopackages

SCENE INVENTORY
------------------------------------------------------------------------


,scene_id,graphml_exists,roads_exists,ready_to_process
0,Ghana_141910,True,True,True
1,India_1018327,True,True,True
2,India_1050276,True,True,True
3,India_1068117,True,True,True
4,India_285297,True,True,True
5,India_383430,True,True,True
6,India_500266,True,True,True
7,India_773682,True,True,True
8,India_804466,True,True,True
9,India_943439,True,True,True



INVENTORY SUMMARY
------------------------------------------------------------------------
GraphML files found     : 26
GeoPackages matched     : 26
Scenes ready to process : 26

All transportation scenes are ready.


In [24]:
# ============================================================
# CELL 17: PROCESS ALL TRANSPORTATION SCENES
# ============================================================

import time
import traceback
import pandas as pd

print("=" * 72)
print("BATCH TRANSPORTATION KNOWLEDGE EXTRACTION")
print("=" * 72)

BATCH_OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "transportation_knowledge"
)

BATCH_ROAD_OUTPUT_DIR = (
    BATCH_OUTPUT_DIR
    / "road_knowledge"
)

BATCH_PROFILE_OUTPUT_DIR = (
    BATCH_OUTPUT_DIR
    / "scene_profiles"
)

BATCH_TOKEN_OUTPUT_DIR = (
    BATCH_OUTPUT_DIR
    / "scene_tokens"
)

BATCH_LOG_OUTPUT_DIR = (
    BATCH_OUTPUT_DIR
    / "logs"
)

for output_dir in [
    BATCH_ROAD_OUTPUT_DIR,
    BATCH_PROFILE_OUTPUT_DIR,
    BATCH_TOKEN_OUTPUT_DIR,
    BATCH_LOG_OUTPUT_DIR,
]:
    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

ready_scenes_df = scene_inventory_df.loc[
    scene_inventory_df["ready_to_process"]
].copy()

batch_results = []
successful_profiles = []
successful_tokens = []

total_scenes = len(ready_scenes_df)
batch_start_time = time.perf_counter()

for scene_number, scene_row in enumerate(
    ready_scenes_df.itertuples(index=False),
    start=1,
):

    scene_id = scene_row.scene_id
    scene_start_time = time.perf_counter()

    print("\n" + "#" * 72)
    print(
        f"SCENE {scene_number}/{total_scenes}: "
        f"{scene_id}"
    )
    print("#" * 72)

    try:

        result = process_transportation_scene(
            scene_id=scene_id,
            graphml_path=scene_row.graphml_path,
            roads_path=scene_row.roads_path,
            road_output_dir=BATCH_ROAD_OUTPUT_DIR,
            profile_output_dir=BATCH_PROFILE_OUTPUT_DIR,
            token_output_dir=BATCH_TOKEN_OUTPUT_DIR,
            save_outputs=True,
        )

        scene_elapsed_seconds = (
            time.perf_counter()
            - scene_start_time
        )

        batch_results.append(
            {
                **result["processing_summary"],
                "processing_seconds":
                    round(scene_elapsed_seconds, 2),
                "error_type":
                    None,
                "error_message":
                    None,
            }
        )

        successful_profiles.append(
            result["scene_profile"]
        )

        successful_tokens.append(
            {
                "scene_id":
                    scene_id,

                "scene_transportation_token":
                    result["scene_token"],
            }
        )

        print(
            f"\nCompleted {scene_id} in "
            f"{scene_elapsed_seconds / 60:.2f} minutes"
        )

    except Exception as error:

        scene_elapsed_seconds = (
            time.perf_counter()
            - scene_start_time
        )

        print(
            f"\nFAILED SCENE: {scene_id}"
        )

        print(
            f"{type(error).__name__}: {error}"
        )

        traceback.print_exc(
            limit=3
        )

        batch_results.append(
            {
                "scene_id":
                    scene_id,

                "status":
                    "FAILED",

                "road_records":
                    None,

                "graph_nodes":
                    None,

                "graph_edges":
                    None,

                "critical_edges":
                    None,

                "single_access_edges":
                    None,

                "bridge_bottlenecks":
                    None,

                "major_road_bottlenecks":
                    None,

                "scene_token_length":
                    None,

                "processing_seconds":
                    round(scene_elapsed_seconds, 2),

                "error_type":
                    type(error).__name__,

                "error_message":
                    str(error),
            }
        )

    # Save the partial log after every scene.
    partial_batch_results_df = pd.DataFrame(
        batch_results
    )

    partial_batch_results_df.to_csv(
        BATCH_LOG_OUTPUT_DIR
        / "transportation_batch_processing_log.csv",
        index=False,
    )

batch_elapsed_seconds = (
    time.perf_counter()
    - batch_start_time
)

batch_results_df = pd.DataFrame(
    batch_results
)

success_count = int(
    (
        batch_results_df["status"]
        == "SUCCESS"
    ).sum()
)

failure_count = int(
    (
        batch_results_df["status"]
        == "FAILED"
    ).sum()
)

print("\n" + "=" * 72)
print("BATCH PROCESSING COMPLETE")
print("=" * 72)

print(
    f"Successful scenes : {success_count}"
)

print(
    f"Failed scenes     : {failure_count}"
)

print(
    f"Total runtime     : "
    f"{batch_elapsed_seconds / 60:.2f} minutes"
)

display(
    batch_results_df
)

BATCH TRANSPORTATION KNOWLEDGE EXTRACTION



########################################################################
SCENE 1/26: Ghana_141910
########################################################################

PROCESSING SCENE: Ghana_141910
Loaded graph        : 2 nodes, 2 edges
Loaded road records : 2
Computing local connectivity for 1 unique undirected pairs...
Processed 1/1

SCENE PROCESSING SUMMARY
------------------------------------------------------------------------
scene_id                      : Ghana_141910
status                        : SUCCESS
road_records                  : 2
graph_nodes                   : 2
graph_edges                   : 1
critical_edges                : 2
single_access_edges           : 2
bridge_bottlenecks            : 0
major_road_bottlenecks        : 0
scene_token_length            : 267

SCENE COMPLETE: Ghana_141910

Completed Ghana_141910 in 0.00 minutes

########################################################################
SCENE 2/26: India_1018327
#############################

Processed 26/26

SCENE PROCESSING SUMMARY
------------------------------------------------------------------------
scene_id                      : India_1018327
status                        : SUCCESS
road_records                  : 54
graph_nodes                   : 25
graph_edges                   : 26
critical_edges                : 12
single_access_edges           : 38
bridge_bottlenecks            : 2
major_road_bottlenecks        : 2
scene_token_length            : 278

SCENE COMPLETE: India_1018327

Completed India_1018327 in 0.00 minutes

########################################################################
SCENE 3/26: India_1050276
########################################################################

PROCESSING SCENE: India_1050276
Loaded graph        : 81 nodes, 186 edges
Loaded road records : 186


Computing local connectivity for 98 unique undirected pairs...
Processed 98/98



SCENE PROCESSING SUMMARY
------------------------------------------------------------------------
scene_id                      : India_1050276
status                        : SUCCESS
road_records                  : 186
graph_nodes                   : 81
graph_edges                   : 97
critical_edges                : 45
single_access_edges           : 71
bridge_bottlenecks            : 10
major_road_bottlenecks        : 4
scene_token_length            : 287

SCENE COMPLETE: India_1050276

Completed India_1050276 in 0.00 minutes

########################################################################
SCENE 4/26: India_1068117
########################################################################

PROCESSING SCENE: India_1068117
Loaded graph        : 51 nodes, 111 edges
Loaded road records : 111
Computing local connectivity for 60 unique undirected pairs...


...

Processed 800/875


Processed 875/875

SCENE PROCESSING SUMMARY
------------------------------------------------------------------------
scene_id                      : USA_170264
status                        : SUCCESS
road_records                  : 1706
graph_nodes                   : 577
graph_edges                   : 872
critical_edges                : 342
single_access_edges           : 203
bridge_bottlenecks            : 6
major_road_bottlenecks        : 2
scene_token_length            : 279

SCENE COMPLETE: USA_170264

Completed USA_170264 in 0.16 minutes

########################################################################
SCENE 24/26: USA_217598
########################################################################

PROCESSING SCENE: USA_217598


Loaded graph        : 51 nodes, 120 edges
Loaded road records : 120
Computing local connectivity for 60 unique undirected pairs...
Processed 60/60



SCENE PROCESSING SUMMARY
------------------------------------------------------------------------
scene_id                      : USA_217598
status                        : SUCCESS
road_records                  : 120
graph_nodes                   : 51
graph_edges                   : 60
critical_edges                : 24
single_access_edges           : 52
bridge_bottlenecks            : 2
major_road_bottlenecks        : 0
scene_token_length            : 287

SCENE COMPLETE: USA_217598

Completed USA_217598 in 0.00 minutes

########################################################################
SCENE 25/26: USA_86502
########################################################################

PROCESSING SCENE: USA_86502
Loaded graph        : 37 nodes, 72 edges
Loaded road records : 72
Computing local connectivity for 37 unique undirected pairs...
Processed 37/37

SCENE PROCESSING SUMMARY
------------------------------------------------------------------------
scene_id                     

,scene_id,status,road_records,graph_nodes,graph_edges,critical_edges,single_access_edges,bridge_bottlenecks,major_road_bottlenecks,scene_token_length,processing_seconds,error_type,error_message
0,Ghana_141910,SUCCESS,2,2,1,2,2,0,0,267,0.09,None,None
1,India_1018327,SUCCESS,54,25,26,12,38,2,2,278,0.08,None,None
2,India_1050276,SUCCESS,186,81,97,45,71,10,4,287,0.22,None,None
3,India_1068117,SUCCESS,111,51,60,23,50,2,12,283,0.13,None,None
4,India_285297,SUCCESS,68,35,40,14,23,3,7,287,0.10,None,None
5,India_383430,SUCCESS,40,20,20,8,32,6,4,278,0.08,None,None
6,India_500266,SUCCESS,134,53,67,28,24,2,2,279,0.14,None,None
7,India_773682,SUCCESS,212,83,106,44,36,0,2,277,0.35,None,None
8,India_804466,SUCCESS,173,90,94,35,92,2,6,284,0.30,None,None
9,India_943439,SUCCESS,10,6,5,2,10,2,4,278,0.26,None,None


In [25]:
# ============================================================
# CELL 18: BUILD MASTER TRANSPORTATION KNOWLEDGE DATASETS
# ============================================================

import json
import pandas as pd
from pathlib import Path

print("=" * 72)
print("BUILDING MASTER TRANSPORTATION KNOWLEDGE DATASETS")
print("=" * 72)

MASTER_OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "transportation_knowledge"
    / "master"
)

MASTER_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ------------------------------------------------------------
# 1. Validate batch completion
# ------------------------------------------------------------

successful_batch_df = batch_results_df.loc[
    batch_results_df["status"] == "SUCCESS"
].copy()

failed_batch_df = batch_results_df.loc[
    batch_results_df["status"] == "FAILED"
].copy()

print(
    f"Successful scenes : {len(successful_batch_df)}"
)

print(
    f"Failed scenes     : {len(failed_batch_df)}"
)

if not failed_batch_df.empty:
    print("\nWARNING: Some scenes failed:")
    display(
        failed_batch_df[
            [
                "scene_id",
                "error_type",
                "error_message",
            ]
        ]
    )

# ------------------------------------------------------------
# 2. Combine scene profiles
# ------------------------------------------------------------

master_scene_profiles_df = pd.concat(
    successful_profiles,
    ignore_index=True,
)

master_scene_profiles_df = (
    master_scene_profiles_df
    .sort_values("scene_id")
    .reset_index(drop=True)
)

master_scene_profiles_csv = (
    MASTER_OUTPUT_DIR
    / "master_scene_transportation_profiles.csv"
)

master_scene_profiles_json = (
    MASTER_OUTPUT_DIR
    / "master_scene_transportation_profiles.json"
)

master_scene_profiles_df.to_csv(
    master_scene_profiles_csv,
    index=False,
)

master_scene_profiles_df.to_json(
    master_scene_profiles_json,
    orient="records",
    indent=2,
)

# ------------------------------------------------------------
# 3. Combine scene tokens
# ------------------------------------------------------------

master_scene_tokens_df = pd.DataFrame(
    successful_tokens
)

master_scene_tokens_df = (
    master_scene_tokens_df
    .sort_values("scene_id")
    .reset_index(drop=True)
)

master_scene_tokens_csv = (
    MASTER_OUTPUT_DIR
    / "master_scene_transportation_tokens.csv"
)

master_scene_tokens_json = (
    MASTER_OUTPUT_DIR
    / "master_scene_transportation_tokens.json"
)

master_scene_tokens_df.to_csv(
    master_scene_tokens_csv,
    index=False,
)

master_scene_tokens_df.to_json(
    master_scene_tokens_json,
    orient="records",
    indent=2,
)

# ------------------------------------------------------------
# 4. Save batch processing summary
# ------------------------------------------------------------

master_batch_summary_csv = (
    MASTER_OUTPUT_DIR
    / "master_transportation_processing_summary.csv"
)

batch_results_df.to_csv(
    master_batch_summary_csv,
    index=False,
)

# ------------------------------------------------------------
# 5. Create compact scene manifest
# ------------------------------------------------------------

scene_manifest_df = successful_batch_df[
    [
        "scene_id",
        "road_records",
        "graph_nodes",
        "graph_edges",
        "critical_edges",
        "single_access_edges",
        "bridge_bottlenecks",
        "major_road_bottlenecks",
        "scene_token_length",
        "processing_seconds",
    ]
].copy()

scene_manifest_df = (
    scene_manifest_df
    .sort_values("scene_id")
    .reset_index(drop=True)
)

scene_manifest_csv = (
    MASTER_OUTPUT_DIR
    / "transportation_scene_manifest.csv"
)

scene_manifest_json = (
    MASTER_OUTPUT_DIR
    / "transportation_scene_manifest.json"
)

scene_manifest_df.to_csv(
    scene_manifest_csv,
    index=False,
)

scene_manifest_df.to_json(
    scene_manifest_json,
    orient="records",
    indent=2,
)

# ------------------------------------------------------------
# 6. Final validation checks
# ------------------------------------------------------------

profile_scene_ids = set(
    master_scene_profiles_df["scene_id"]
)

token_scene_ids = set(
    master_scene_tokens_df["scene_id"]
)

manifest_scene_ids = set(
    scene_manifest_df["scene_id"]
)

inventory_scene_ids = set(
    ready_scenes_df["scene_id"]
)

all_scene_sets_match = (
    profile_scene_ids
    == token_scene_ids
    == manifest_scene_ids
    == inventory_scene_ids
)

duplicate_profile_count = int(
    master_scene_profiles_df["scene_id"]
    .duplicated()
    .sum()
)

duplicate_token_count = int(
    master_scene_tokens_df["scene_id"]
    .duplicated()
    .sum()
)

missing_token_count = int(
    master_scene_tokens_df[
        "scene_transportation_token"
    ]
    .isna()
    .sum()
)

print("\nVALIDATION SUMMARY")
print("-" * 72)

print(
    f"Master profile records  : "
    f"{len(master_scene_profiles_df)}"
)

print(
    f"Master token records    : "
    f"{len(master_scene_tokens_df)}"
)

print(
    f"Manifest records        : "
    f"{len(scene_manifest_df)}"
)

print(
    f"Scene ID sets match     : "
    f"{all_scene_sets_match}"
)

print(
    f"Duplicate profiles      : "
    f"{duplicate_profile_count}"
)

print(
    f"Duplicate tokens        : "
    f"{duplicate_token_count}"
)

print(
    f"Missing tokens          : "
    f"{missing_token_count}"
)

print("\nSAVED MASTER FILES")
print("-" * 72)

for output_path in [
    master_scene_profiles_csv,
    master_scene_profiles_json,
    master_scene_tokens_csv,
    master_scene_tokens_json,
    master_batch_summary_csv,
    scene_manifest_csv,
    scene_manifest_json,
]:
    print(output_path)

print("\nMASTER SCENE PROFILE PREVIEW")
display(
    master_scene_profiles_df.head()
)

print("\nMASTER SCENE TOKEN PREVIEW")
display(
    master_scene_tokens_df.head()
)

print("=" * 72)

BUILDING MASTER TRANSPORTATION KNOWLEDGE DATASETS
Successful scenes : 26
Failed scenes     : 0



VALIDATION SUMMARY
------------------------------------------------------------------------
Master profile records  : 26
Master token records    : 26
Manifest records        : 26
Scene ID sets match     : True
Duplicate profiles      : 0
Duplicate tokens        : 0
Missing tokens          : 0

SAVED MASTER FILES
------------------------------------------------------------------------
/home/adjeiowusu1/myproject/ResilientVLM/data/processed/transportation_knowledge/master/master_scene_transportation_profiles.csv
/home/adjeiowusu1/myproject/ResilientVLM/data/processed/transportation_knowledge/master/master_scene_transportation_profiles.json
/home/adjeiowusu1/myproject/ResilientVLM/data/processed/transportation_knowledge/master/master_scene_transportation_tokens.csv
/home/adjeiowusu1/myproject/ResilientVLM/data/processed/transportation_knowledge/master/master_scene_transportation_tokens.json
/home/adjeiowusu1/myproject/ResilientVLM/data/processed/transportation_knowledge/master/master_tra

,scene_id,road_record_count,graph_node_count,graph_edge_count,connected_component_count,largest_component_nodes,largest_component_share,major_road_count,major_road_share,high_capacity_road_count,...,high_topological_vulnerability_count,high_topological_vulnerability_share,mean_edge_criticality_score,mean_topological_vulnerability_score,road_class_distribution,hierarchy_group_distribution,network_role_distribution,redundancy_distribution,vulnerability_distribution,scene_transportation_token
0,Ghana_141910,2,2,1,1,2,1.0,0,0.0000,0,...,2,1.0000,0.3292,0.6981,{'unclassified': 2},{'local': 2},{'Critical Corridor': 2},{'None': 2},{'High': 2},<NetworkConnectivity:High> <SingleAccessExposu...
1,India_1018327,54,25,26,1,25,1.0,4,0.0741,0,...,12,0.2222,0.4140,0.6128,"{'unclassified': 22, 'tertiary': 14, 'resident...","{'local': 36, 'local_collector': 14, 'arterial...","{'Single Access': 28, 'Local Access': 14, 'Cri...","{'None': 38, 'Low': 16}","{'High': 36, 'Low': 16, 'Very High': 2}",<NetworkConnectivity:High> <SingleAccessExposu...
2,India_1050276,186,81,97,1,81,1.0,27,0.1452,7,...,38,0.2043,0.3876,0.4523,"{'unclassified': 64, 'residential': 64, 'terti...","{'local': 128, 'local_collector': 31, 'arteria...","{'Local Access': 86, 'Single Access': 53, 'Pri...","{'Low': 82, 'None': 71, 'Moderate': 33}","{'Low': 109, 'High': 65, 'Moderate': 4, 'Very ...",<NetworkConnectivity:High> <SingleAccessExposu...
3,India_1068117,111,51,60,1,51,1.0,27,0.2432,27,...,28,0.2523,0.4290,0.5087,"{'unclassified': 64, 'trunk': 23, 'residential...","{'local': 84, 'major_arterial': 27}","{'Local Access': 48, 'Single Access': 40, 'Pri...","{'Low': 52, 'None': 50, 'Moderate': 9}","{'Low': 49, 'High': 38, 'Very High': 12, 'Mode...",<NetworkConnectivity:High> <SingleAccessExposu...
4,India_285297,68,35,40,1,35,1.0,20,0.2941,20,...,15,0.2206,0.4416,0.4612,"{'unclassified': 46, 'trunk': 14, 'trunk_link'...","{'local': 46, 'major_arterial': 20, 'local_col...","{'Local Access': 35, 'Single Access': 19, 'Pri...","{'Low': 33, 'None': 23, 'Moderate': 12}","{'Low': 36, 'High': 18, 'Moderate': 9, 'Very H...",<NetworkConnectivity:High> <SingleAccessExposu...



MASTER SCENE TOKEN PREVIEW


,scene_id,scene_transportation_token
0,Ghana_141910,<NetworkConnectivity:High> <SingleAccessExposu...
1,India_1018327,<NetworkConnectivity:High> <SingleAccessExposu...
2,India_1050276,<NetworkConnectivity:High> <SingleAccessExposu...
3,India_1068117,<NetworkConnectivity:High> <SingleAccessExposu...
4,India_285297,<NetworkConnectivity:High> <SingleAccessExposu...
